# Reproducing Mantas, Contestabile, Skara et al. (2024) from dandiset 001828

Paper. Mantas I, Contestabile A, Skara V, et al. (2024). Anxa1+ dopamine neuron vulnerability
defines prodromal Parkinson's disease bradykinesia and procedural motor learning impairment.
bioRxiv, [doi:10.1101/2024.12.22.629963](https://doi.org/10.1101/2024.12.22.629963). The claims file
([claims/001828/10.1101-2024.12.22.629963.yaml](../../claims/001828/10.1101-2024.12.22.629963.yaml))
was written from the full text of preprint version 1 (Results, Methods, legends of Figures 1 to 6);
the supplementary figures were not in the text read.

Dandiset. [DANDI:001828](https://dandiarchive.org/dandiset/001828/0.261020.1002), version
0.261020.1002 (doi:10.48324/dandi.001828/0.261020.1002). The paper cites no version of the dandiset
(it was created after the preprint). This is the only published version, so it is pinned; the draft
holds the same 1014 assets (checked below).

What the dandiset holds. Behaviour and fiber photometry only: open field sessions with
DeepLabCut pose, kinematics and VAME motif labels (Figures 4 and 5), optogenetically evoked dLight
release (Figure S2), water drop sessions with photometry (Figure 3D to 3G), forelimb reaching with
annotated reaches (Figure 6E to 6J) and arrow maze sessions with pose (Figure 6A to 6D). There is no
histology, RNA-seq, qPCR or nosepoke data, and the arrow maze has no end-zone entries or rewards.

Outside inputs. GEO GSE282753, the paper's RNA-seq deposit, read for fig1b only: its count table
(`GSE282753_counts_mitopark.txt.gz`, md5 printed in the data check) holds the MitoPark half of what
the claim needs, and that half is run.

| Claim | Status here |
|---|---|
| figs2-evoked-da-dcp-ccp | tested |
| fig3d-anxa1-ramp-to-touch, fig3e-dlight-dcp-ccp-reward, fig3fg-auc-change-from-baseline | tested |
| fig4b-speed-correlation-dcp, fig4b-speed-correlation-ccp-negative | tested |
| fig5c-tetx-locomotion, fig5e-early-models-locomotion | tested |
| fig6c-maze-locomotion, fig6j-reaching-accuracy | tested |
| fig4c-vame-motifs, fig4e-anxa1-high-speed-motifs, fig4f-auc-correlations, fig4h-auc-motif-speed, fig5g-pc1-stage, fig5i-pc1-motif-speed | blocked: the paper's 37-motif labelling is not in the dandiset; the same analyses on the dandiset's own labels are shown, not deciding |
| fig5h-walk-stationary-loadings, fig5j-tetx-five-motifs, fig5j-shared-and-stage-motifs, fig5m-walk-motif-network | blocked: as above, and the motif categories and numbers are the paper's |
| fig6d-maze-preference-learning, fig6d-maze-efficiency | blocked: no end-zone entries or rewards |
| fig1b-anxa1-downregulated-rnaseq | blocked: GSE282753 lacks the cMfn2KO data (data elsewhere); the MitoPark half is run |
| fig1c, fig1e, fig1h, fig1j, fig1m, figs1 (two), fig1p, fig2c, fig2f, fig2i, fig2n | blocked: histology and qPCR not in the dandiset |
| fig3i, fig3kl, fig3o | blocked: nosepoke counts not in the dandiset |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

import anxa1

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = anxa1.list_assets()
nwb = assets[assets.path.str.endswith(".nwb")].copy()
nwb["kind"] = np.select(
    [nwb.path.str.endswith("behavior+image.nwb"),
     nwb.path.str.contains("ses-oft") & ~nwb.path.str.contains("behavior"),
     nwb.path.str.contains("ses-reaching"),
     nwb.path.str.contains("ses-tmaze") & nwb.path.str.endswith("_image.nwb")],
    ["open field", "evoked release", "reaching", "arrow maze"], "other")
print(f"dandiset {anxa1.DANDISET_ID} version {anxa1.DANDISET_VERSION}: {len(assets)} assets, "
      f"{len(nwb)} NWB files, {(~assets.path.str.endswith('.nwb')).sum()} external videos")
print(nwb.kind.value_counts().to_string())

def load(kind, fn):
    sel = nwb[nwb.kind == kind]
    out = anxa1.load_many(fn, list(sel.asset_id))
    for s, p in zip(out, sel.path):
        s["path"] = p
    return out

t0 = time.time(); oft = load("open field", anxa1.load_oft); print(f"open field {len(oft)} files, {time.time() - t0:.0f} s")
t0 = time.time(); opto = load("evoked release", anxa1.load_opto); print(f"evoked release {len(opto)} files, {time.time() - t0:.0f} s")
t0 = time.time(); reach = load("reaching", anxa1.load_reach); print(f"reaching {len(reach)} files, {time.time() - t0:.0f} s")
t0 = time.time(); maze = load("arrow maze", anxa1.load_maze); print(f"arrow maze {len(maze)} files, {time.time() - t0:.0f} s")

## Data check

Each check of the metadata against the data is stated under its output.

In [ ]:
meta = anxa1.version_metadata()
summary = meta["assetsSummary"]
print("Version DOI:", meta.get("doi"), "| assetsSummary: files", summary["numberOfFiles"], "subjects",
      summary["numberOfSubjects"], "| variableMeasured", summary.get("variableMeasured"))
print("Related resources:", [(r.get("identifier"), r.get("relation")) for r in meta.get("relatedResource", [])])
import json, urllib.request
draft, url = [], f"{anxa1.API}/dandisets/{anxa1.DANDISET_ID}/versions/draft/assets/?page_size=1000"
while url:
    page = json.load(urllib.request.urlopen(url))
    draft += [(a["path"], a["size"]) for a in page["results"]]
    url = page["next"]
pinned = set(zip(assets.path, assets["size"]))
print(f"Draft: {len(draft)} assets; same path and size as the pinned version: {len(pinned & set(draft))}; "
      f"only in draft: {len(set(draft) - pinned)}; only in pinned: {len(pinned - set(draft))}")
subjects = {s["subject"] for s in oft + opto + reach + maze}
print("Distinct subject ids over all NWB files:", len(subjects))

The draft holds the same assets as the pinned version. The archive's summary counts 179 subjects,
which matches the distinct subject ids in the files. Its `variableMeasured` lists only
`ProcessingModule` and `BehavioralTimeSeries`, though the files also hold fiber photometry.

In [ ]:
odf = pd.DataFrame([{
    "subject": s["subject"], "genotype": s["genotype"], "sex": s["sex"], "experiment": s["experiment"],
    "group_raw": s["group_raw"], "group": s["group"], "session_id": s["session_id"][:6],
    "frames": len(s["lin"]), "pose_frames": s["n_pose"], "fp": s["fp"], "start": s["start"][:19]} for s in oft])
print("Open field files by experiment and group (from session_description), and the prefix of session_id:")
print(pd.crosstab([odf.experiment, odf.group_raw, odf.group], odf.session_id).to_string())
paper_n = {"Ctrl-Tdtomato": 18, "Anxa1-TeTx": 14, "Ctrl-Asc Acid": 9, "6OHDA-low": 11,
           "Ctrl-early": 14, "MP-early": 13, "Ctrl-middle": 15, "MP-middle": 14}   # legend of Figure 5
counts = odf[odf.group.isin(anxa1.FIG5_GROUPS)].groupby("group").agg(files=("subject", "size"), mice=("subject", "nunique"))
counts["legend of Figure 5"] = pd.Series(paper_n)
print("\nFigure 5 groups:"); print(counts.loc[anxa1.FIG5_GROUPS].to_string())
print("\nFrames per file (open field without photometry):", odf[~odf.fp].frames.value_counts().to_dict(),
      "| pose frames minus kinematics frames:", (odf.pose_frames - odf.frames).value_counts().to_dict())
print("Mice with more than one open field file:")
multi = odf.groupby("subject").filter(lambda g: len(g) > 1).sort_values(["subject", "start"])
print(multi[["subject", "experiment", "group_raw", "start", "frames", "fp"]].to_string(index=False))

The groups of Figure 5 are all present, with exactly the legend's numbers of mice. 84 of the 160
open field files are named `ses-tmaze-...`, and these and the two files without a session in their
name have session_id `tmaze-...` (86 in the table), although their descriptions, contents (VAME
labels, kinematics, the open field DeepLabCut model) and frame counts are those of open field
sessions. Three experiments are not in the
preprint (`oft_6ohda_tailStriatum`, `oft_anti_anxa1`) and are left out. Sessions without photometry
have 9000 pose frames (5 min at 30 fps); kinematics and motif labels have 30 frames fewer, the VAME
time window. Six mice were recorded both as MP-early and as MP-middle, at two ages; the paper's
groups treat them as separate samples, and so does this notebook.

In [ ]:
lin_units = {s["lin_unit"] for s in oft}
print("linear_velocity unit and conversion:", lin_units, "-> stored values are cm/s")
allv = np.concatenate([s["lin"] for s in oft]); alla = np.concatenate([s["ang"] for s in oft])
print(f"linear speed (cm/s): median {np.nanmedian(allv):.2f}, 99th percentile {np.nanpercentile(allv, 99):.1f}, max {np.nanmax(allv):.1f}; "
      f"NaN fraction {np.isnan(allv).mean():.5f}")
print(f"angular speed (rad/s): median {np.nanmedian(alla):.3f}, 99th percentile {np.nanpercentile(alla, 99):.2f}, min {np.nanmin(alla):.2f}")
ratio = [np.nanmedian(s["lin"][s["dist"] > 0] / s["dist"][s["dist"] > 0]) for s in oft]
print("linear_velocity / distance_moved per file: min", np.round(min(ratio), 4), "max", np.round(max(ratio), 4), "(30 = frames per s)")
scale = []
for s in oft:
    st, d = s["pose_step6"][:len(s["dist"])], s["dist"]
    ok = np.isfinite(st) & np.isfinite(d) & (d > 0.05)
    scale.append(np.median(st[ok] / d[ok]) if ok.sum() > 100 else np.nan)
print(f"pose step (mean of six points, px) / distance_moved (cm), median over {np.isfinite(scale).sum()} files {np.nanmedian(scale):.1f} px per cm "
      f"(5th to 95th percentile {np.nanpercentile(scale, 5):.1f} to {np.nanpercentile(scale, 95):.1f}); "
      f"camera 800 px for a 40 cm box gives 20")
print("pose x/y range (px):", round(min(s["pose_range"][0] for s in oft)), "to", round(max(s["pose_range"][1] for s in oft)))
hashes = pd.Series([hash(s["lin"][:3000].tobytes()) for s in oft])
print("Files sharing the first 3000 speed samples with another file:", int(hashes.duplicated(keep=False).sum()))
motif_max = max(s["motif"].max() for s in oft); motif_min = min(s["motif"].min() for s in oft)
print("Motif labels: range", motif_min, "to", motif_max)

The stored speeds are cm/s (unit m/s with conversion 0.01), equal to `distance_moved` times 30
frames per s, with plausible values for a mouse. Their ratio to the pose displacement is close to
the 20 pixels per cm that an 800 pixel image of a 40 cm box gives, so the scale is consistent with
the data. No two files share a speed trace.

In [ ]:
fp = [s for s in oft if s["fp"]]
rows = []
for s in fp:
    z, off = anxa1.z_on_frames(s)
    tz = s["z_t0"] + np.arange(len(z)) / s["z_rate"]
    smooth = np.convolve(s["dff"], np.ones(12) / 12, mode="same")      # 0.2 s running mean at 60 Hz
    match = {lag: np.corrcoef(z, np.interp(tz + lag, s["dff_t"], smooth))[0, 1] for lag in np.round(np.arange(-0.3, 0.31, 0.05), 2)}
    runs = np.diff(np.r_[0, np.flatnonzero(np.diff(z) != 0) + 1, len(z)])
    zz, v, _ = anxa1.frame_aligned(s)
    ok = np.isfinite(zz) & np.isfinite(v)
    lags = np.arange(-90, 91, 3)
    n = ok.sum(); a, b = zz[ok], v[ok]
    cc = [np.corrcoef(a[max(0, -l):n - max(0, l)], b[max(0, l):n - max(0, -l)])[0, 1] for l in lags]
    rows.append({"group": s["group"], "subject": s["subject"], "start": s["start"][:16], "location": s["location"],
                 "indicator": s["indicator"], "z_samples": len(z), "z_minutes": len(z) / 30 / 60,
                 "frames": len(s["lin"]), "first_frame": off, "z_mean": np.nanmean(z), "z_sd": np.nanstd(z),
                 "r_raw_dff": np.corrcoef(z, np.interp(tz, s["dff_t"], s["dff"]))[0, 1],
                 "r_smoothed_dff": max(match.values()), "at_lag_s": max(match, key=match.get),
                 "held_first": runs[0], "held_last": runs[-1],
                 "lag_of_peak_|r|_s": lags[np.argmax(np.abs(cc))] / 30, "peak_|r|": np.max(np.abs(cc))})
fpdf = pd.DataFrame(rows).sort_values(["group", "subject", "start"])
print(fpdf.round(3).to_string(index=False))
print("\nRecordings and mice per group (legend of Figure 4B: Anxa1 9 mice 9 recordings, DAT 5 and 7, dLight-dCP 6 and 8, dLight-cCP 6 and 7):")
print(fpdf.groupby("group").agg(recordings=("subject", "size"), mice=("subject", "nunique")).to_string())
near = fpdf["lag_of_peak_|r|_s"].abs() <= 0.5
print(f"Recordings whose speed-signal correlation peaks within 0.5 s of zero lag: {near.sum()} of {len(fpdf)} "
      f"(median peak |r| {fpdf[near]['peak_|r|'].median():.3f}); the others: median peak |r| {fpdf[~near]['peak_|r|'].median():.3f}")

The `FiberPhotometrySeriesZScored` series of each open field recording (mean 0, s.d. 1) is not the
dF/F at the same times z-scored: it correlates with the raw dF/F at 0.42 to 0.92, and better with a
0.2 s running mean of it, best at the lag given (`at_lag_s`, the z sample matching the dF/F that
much earlier). Its first 7 or 8 samples, and in most files its last 8, are held constant, the mark of
a centred smoothing window of about half a second. The series is described as session-level z-scored and aligned to the
behaviour frames, but its smoothing and the 0.1 s lag are not documented. It is used as the
dandiset's processed trace, put on the frames by its `starting_time` (`first_frame`). Seven
recordings hold under 13 minutes of photometry, against the 20 minute sessions of the Methods. The
lag at which speed and signal correlate most is within 0.5 s of zero in 16 of 31 recordings; the
correlations are weak at every lag (median peak |r| 0.14 and 0.11), so this check is weak. The photometry and the behaviour frames are consistent with one clock where it
can be checked, not shown to share it in every recording. The recording counts differ
from the legend's by one or two in three groups (Anxa1 10 against 9, DAT 6 against 7, dLight-dCP 5
mice against 6), and the dandiset does not say which recordings the paper used; all are used here.

In [ ]:
SHIFTS = list(range(-45, 46, 3))
krows = []
for s in oft:
    st, d = s["pose_step6"], s["dist"]
    cl = {}
    for l in range(0, 31):          # kinematics frame i against pose frame i + l
        a = st[l:l + len(d)]; ok = np.isfinite(a) & np.isfinite(d)
        cl[l] = np.corrcoef(a[ok], d[ok])[0, 1]
    fit = anxa1.label_speed_fit(s["lin"], s["motif"], SHIFTS)
    krows.append({"subject": s["subject"], "group": s["group"], "path": s["path"], "fp": s["fp"], "kin_t0": s["kin_t0"], "motif_t0": s["motif_t0"], "pose minus kinematics frames": s["n_pose"] - len(s["lin"]),
                  "motif minus kinematics frames": len(s["motif"]) - len(s["lin"]),
                  "kinematics lag (pose frames)": max(cl, key=cl.get), "best motif shift": max(fit, key=fit.get),
                  **{f"eta2 {k:+d}": fit[k] for k in (-30, -15, 0, 15, 18, 30)}, "eta2 best": max(fit.values())})
KT = pd.DataFrame(krows)
for col in ["kin_t0", "motif_t0", "pose minus kinematics frames", "motif minus kinematics frames", "kinematics lag (pose frames)", "best motif shift"]:
    print(f"{col}: {KT[col].value_counts().sort_index().to_dict()}")
print("Speed variance explained by the motif label (eta squared), median over files, by shift of the labels (frames, + later):")
print(KT.filter(like="eta2").median().round(3).to_dict())
BEST_SHIFT = int(KT["best motif shift"].median())
print("Median best shift:", BEST_SHIFT, "frames")
zt = pd.DataFrame([{"group": s["group"], "subject": s["subject"], "z_t0 (s)": s["z_t0"], "dF/F first timestamp (s)": s["dff_t"][0],
                    "first frame": anxa1.z_on_frames(s)[1]} for s in fp]).sort_values(["group", "subject"])
zt["z_t0 minus dF/F start"] = zt["z_t0 (s)"] - zt["dF/F first timestamp (s)"]
offk = KT[KT["kinematics lag (pose frames)"] != 0]
print(f"\nFiles whose kinematics are not at lag 0 to the pose: {len(offk)}, of which with photometry: {int(offk.fp.sum())}")
print(offk[["subject", "group", "fp", "kinematics lag (pose frames)"]].sort_values(["fp", "group"]).to_string(index=False))
KLAG = dict(zip(KT.path, KT["kinematics lag (pose frames)"]))
print("\nPhotometry start per recording:"); print(zt.round(3).to_string(index=False))


In every open field file the kinematics and the motif labels start at t = 0, as the pose does, and
both have 30 frames fewer than the pose. The kinematics match the pose with no lag (the pose step
correlates best with `distance_moved` at lag 0 in the count shown), so they are on the pose frames
and the 30 missing frames are at the end. The motif labels are not: speed is explained best by the
labels moved later by the median shift printed, about half the 30-frame VAME window, so a stored
label describes the window that starts at its frame. The files say nothing of this; motif onsets as
stored come early by that much. This bears on the motif-onset AUCs (shown below with the labels
moved), not on the speed correlations of fig4b, which use the kinematics and the photometry, whose
`starting_time` matches its own dF/F timestamps. Of the 19 files whose kinematics are not at lag 0
to the pose, five hold photometry (listed); fig4b is also run with their speed moved by that lag.

In [ ]:
orows = []
for s in opto:
    z = anxa1.zscore(s["dff"])
    g, P, kept = anxa1.peri(s["t"], z, s["on"], -1, 1)
    gs, Ps, _ = anxa1.peri(s["t"], z, s["on"] + 2.0, -1, 1)
    resp = lambda Q, gg: (Q[:, (gg >= 0) & (gg <= 1)].mean(1) - Q[:, (gg >= -1) & (gg < 0)].mean(1)).mean()
    orows.append({"subject": s["subject"], "genotype": s["genotype"], "location": s["location"], "indicator": s["indicator"],
                  "power_mW": float(np.unique(s["power"])[0]), "n_powers": len(np.unique(s["power"])),
                  "trains": len(s["on"]), "kept": len(kept), "train_s": float(np.median(s["off"] - s["on"])),
                  "pulses_median": float(np.median(s["pulses"])), "period_ms": float(np.unique(s["period_ms"])[0]),
                  "onset_interval_s": float(np.median(np.diff(s["on"]))), "dff_hz": 1 / np.median(np.diff(s["t"])),
                  "minutes": (s["t"][-1] - s["t"][0]) / 60, "response_at_onset": resp(P, g), "response_2s_later": resp(Ps, gs)})
optodf = pd.DataFrame(orows).sort_values(["location", "subject", "power_mW"])
print(optodf.round(3).to_string(index=False))

The evoked release sessions are four Anxa1-Flp mice with dLight1.3b, two with the fiber in dCP and
two in cCP, each at five powers in separate sessions of about 20 minutes, with 1 s trains of 40
pulses at 40 Hz every 4 s (some trains are cut short, with 1 to 39 pulses). The dF/F rises after
train onsets and not 2 s later, so the stimulation table and the photometry share a clock. The
session descriptions call these self-stimulation sessions in which the mouse nosepoked, but the
trains come at fixed 4 s intervals. All four mice are listed as WT in their open field files and as
Anxa1-flp here (checked below).

In [ ]:
rdf = pd.DataFrame([{"subject": s["subject"], "genotype": s["genotype"], "experiment": s["experiment"], "group": s["group"],
                     "phase": s["phase"], "fp": s["fp"], "fp_group": s.get("fp_group", ""), "start": s["start"][:16],
                     "events": len(s["events"]), "video_hz": s["video_rate"]} for s in reach])
print("Reaching files by experiment, group and test phase:")
print(rdf.groupby(["experiment", "group", "phase"]).agg(files=("subject", "size"), mice=("subject", "nunique"),
                                                       with_events=("events", lambda e: int((e > 0).sum()))).to_string())
allev = pd.concat([s["events"].assign(subject=s["subject"], fp=s["fp"]) for s in reach], ignore_index=True)
print("\nEvents by label and paw:"); print(pd.crosstab(allev.label, [allev.fp, allev.paw]).to_string())
print("max |event time - frame / video rate| (s):",
      max(float(np.max(np.abs(s["events"].time - s["events"].frame / s["video_rate"]))) for s in reach if len(s["events"])))
d = allev.dropna(subset=["distance_cm"])
print(f"distance_cm: {len(d)} events, range {d.distance_cm.min():.3f} to {d.distance_cm.max():.3f} cm; "
      f"distance_px / reference_px / distance_cm: min {np.min(d.distance_px / d.reference_px / d.distance_cm):.3f}, "
      f"max {np.max(d.distance_px / d.reference_px / d.distance_cm):.3f}")
print("Events without a distance:", allev.distance_cm.isna().groupby(allev.fp).sum().to_dict(), "(by photometry session)")
print("Events in photometry sessions:", int(allev.fp.sum()), "| made with the mouth:", int((allev.fp & (allev.paw == "mouth")).sum()))

Event times equal the annotated video frame divided by 60, the video rate. Distances in cm are the
pixel distance divided by the session's reference length in pixels, a 1 cm reference, so they are
in cm as stated. In the photometry sessions most events (746 of 1089) are made with the mouth, and 673 carry
no distance; these are the water drop sessions of Figure 3, stored as reaching sessions. Some
sessions of the 6-OHDA tail striatum cohort hold hundreds of `unknown` events; that cohort is not in
the preprint.

In [ ]:
rng_check = np.random.default_rng(11)
frows = []
for s in [s for s in reach if s["fp"]]:
    e = s["events"][s["events"].label == "correct"].time.values
    z = anxa1.zscore(s["dff"])
    g, P, kept = anxa1.peri(s["t"], z, e, -2, 2)
    fake = rng_check.uniform(s["t"][0] + 3, s["t"][-1] - 3, 200)
    gf, Pf, _ = anxa1.peri(s["t"], z, fake, -2, 2)
    mod = lambda Q, gg: np.ptp(Q.mean(0)) if len(Q) else np.nan
    frows.append({"group": s["fp_group"], "subject": s["subject"], "genotype": s["genotype"], "location": s["location"],
                  "indicator": s["indicator"], "start": s["start"][:16], "minutes": (s["t"][-1] - s["t"][0]) / 60,
                  "correct": len(e), "kept": len(kept), "other_events": int((s["events"].label != "correct").sum()),
                  "range_of_mean_at_events": mod(P, g), "range_at_random_times": mod(Pf, gf)})
wdf = pd.DataFrame(frows).sort_values(["group", "subject", "start"])
print(wdf.round(3).to_string(index=False))
print("\nRecordings and mice per group (legend of Figure 3D-E: Anxa1 7 mice 10 recordings, DAT 3 and 4, dLight-dCP 6 and 13, dLight-cCP 6 and 11):")
print(wdf.groupby("group").agg(recordings=("subject", "size"), mice=("subject", "nunique")).to_string())

The mean signal around the annotated drop retrievals varies more than around 200 random times in
every recording, so events and photometry are consistent with one clock. Part of the contrast is the
noise of a mean over few events, so it cannot tell in sessions with 1 to 3 events. The dandiset holds more recordings than the legend counts
in every group, and does not say which were used. The session descriptions say the GCaMP recordings
are of SN cell bodies, while the fiber table and the indicator descriptions place them in dCP, as the
paper does; the fiber table is followed here. The group of each recording is taken from its
indicator, line and fiber site.

In [ ]:
mdf = pd.DataFrame([{"subject": s["subject"], "cohort": s["cohort"], "group": s["group"], "day": s["day"],
                     "frames": s["xy"].shape[1], "rate": s["rate"], "x_min": np.nanmin(s["xy"][..., 0]), "x_max": np.nanmax(s["xy"][..., 0]),
                     "tail_x_p1": np.nanpercentile(s["xy"][0, :, 0], 1), "tail_x_p99": np.nanpercentile(s["xy"][0, :, 0], 99),
                     "tail_y_p1": np.nanpercentile(s["xy"][0, :, 1], 1), "tail_y_p99": np.nanpercentile(s["xy"][0, :, 1], 99),
                     "tail_conf_below_0.9": float((s["conf"][0] < 0.9).mean()),
                     "tail_steps_over_30px": float((np.hypot(*np.diff(s["xy"][0], axis=0).T) > 30).mean())} for s in maze])
print(mdf.groupby(["cohort", "group", "day"]).agg(files=("subject", "size"), mice=("subject", "nunique")).to_string())
print("frames:", mdf.frames.value_counts().to_dict(), "at", mdf.rate.unique(), "Hz ->", mdf.frames.iloc[0] / mdf.rate.iloc[0] / 60, "min")
print(f"x range {mdf.x_min.min():.0f} to {mdf.x_max.max():.0f} px; tail base confidence below 0.9 in "
      f"{mdf['tail_conf_below_0.9'].median():.3f} of frames (median), steps over 30 px in {mdf.tail_steps_over_30px.median():.4f}")
print("Arrow maze mice (legend of Figure 6C: Ctrl-Tdtomato 18, Anxa1-TeTx 10):",
      mdf[mdf.cohort == "tmaze_anxa1_tet"].groupby("group").subject.nunique().to_dict())
print("\nTail base position (px), 1st and 99th percentile per session, median over the sessions of each day (TeTx cohort);")
print("the arena's extent in pixels, and so the pixel scale, should not change between days:")
print(mdf[mdf.cohort == "tmaze_anxa1_tet"].groupby(["day", "group"])[["tail_x_p1", "tail_x_p99", "tail_y_p1", "tail_y_p99"]].median().round(0).to_string())


The arrow maze sessions last 10 minutes at 30 fps, as in the Methods, and each mouse has four
days. The TeTx cohort has 17 controls against the legend's 18. The files hold pose only: no
end-zone entries, rewards or arena geometry, so the transitions and rewards of Figure 6D cannot be
counted. About 0.2% of tail base steps are jumps of more than 30 pixels (1.5 cm in 33 ms), which a
mean speed is sensitive to.

In [ ]:
geno = pd.concat([odf[["subject", "genotype"]].assign(kind="open field"),
                  pd.DataFrame([{"subject": s["subject"], "genotype": s["genotype"], "kind": "evoked release"} for s in opto]),
                  rdf[["subject", "genotype"]].assign(kind="reaching"),
                  pd.DataFrame([{"subject": s["subject"], "genotype": s["genotype"], "kind": "arrow maze"} for s in maze])])
geno["g"] = geno.genotype.str.lower().str.replace("_", "-")
conflict = geno.groupby("subject").filter(lambda x: x.g.nunique() > 1)
print("Subjects with different genotypes in different files:")
print(conflict.drop_duplicates(["subject", "genotype", "kind"]).sort_values("subject").to_string(index=False))
tet_oft = odf[odf.experiment == "oft_Tetx"].set_index("subject").group
tet_reach = rdf[rdf.experiment == "reaching_test_anxa1_tet"].drop_duplicates("subject").set_index("subject").group
both = tet_oft.index.intersection(tet_reach.index)
cmp = pd.DataFrame({"open field": tet_oft[both], "reaching": tet_reach[both]})
cmp["agree"] = (cmp["open field"] == "Ctrl-Tdtomato") == (cmp["reaching"] == "Ctrl")
print("\nTeTx mice in both the open field and the reaching cohort:"); print(cmp.to_string())

Subject genotypes are not consistent across files: mice with dLight and Anxa1-Flp ChRmine are WT in
their open field files and Anxa1-flp elsewhere, and one mouse (687662) is anxa1-flp in the open
field and Dat_cre in reaching. One mouse (779575) is a control in the open field TeTx experiment and
a TeTx mouse in the reaching cohort. The analysis takes each file's own group, as the paper's
groups are per experiment, and records these as findings.

In [ ]:
import re
orows = []
for kind, files in [("open field", oft), ("evoked release", opto), ("reaching", reach), ("arrow maze", maze)]:
    c = pd.Series([re.sub(r"^acquisition/Video .*", "acquisition/Video <session>", g) for s in files for g in s["groups"]]).value_counts()
    orows += [{"kind": kind, "object": g, "files": f"{n} of {len(files)}"} for g, n in c.items()]
print("Objects under acquisition/, intervals/ and each processing module, by file kind:")
print(pd.DataFrame(orows).to_string(index=False))
allobj = {g for s in oft + opto + reach + maze for g in s["groups"]}
print("Objects whose name mentions a poke, port, reward, zone, entry or trial:",
      sorted(g for g in allobj if re.search("poke|port|reward|zone|entr|trial", g, re.I)) or "none")


No file holds a nosepoke, reward, end-zone or trial table: the evoked release files hold the
photometry and the stimulation epochs, the arrow maze files hold pose and the external video only,
and the reaching files hold the annotated events, photometry or a pose model. The self-stimulation,
ratio-task and maze-learning data of Figures 3I to 3O and 6D are therefore not in the dandiset.

In [ ]:
GEO_FILES, GEO_SERIES = anxa1.geo_supplementary_files()
print("GEO GSE282753 supplementary files:"); print(GEO_FILES.to_string(index=False))
print("Series:", {k: (v[:160] + " ...") if isinstance(v, str) and len(v) > 160 else v for k, v in GEO_SERIES.items()})
GEO, GEO_LIBS, GEO_MD5 = anxa1.load_geo_counts()
print("GEO GSE282753 counts file:", anxa1.GEO_COUNTS.rsplit("/", 1)[1], "| md5", GEO_MD5, "|", GEO.shape[0], "genes,", GEO.shape[1], "libraries")
print(GEO_LIBS.groupby("group").agg(mice=("mouse", "nunique"), libraries=("library", "size")).to_string())
print("Libraries per mouse:", GEO_LIBS.groupby("mouse").size().to_dict())
lib_size = GEO.sum()
print(f"Counts are integers: {bool((GEO.values == GEO.values.round()).all())}, min {GEO.values.min()}; "
      f"library size {lib_size.min():,.0f} to {lib_size.max():,.0f} reads")
print("Anxa1 counts summed per mouse:", GEO.loc["Anxa1"].groupby(GEO_LIBS.mouse.values).sum().to_dict())

GEO GSE282753, the paper's only RNA-seq deposit, holds one count table: MitoPark and control DANs at
10 weeks, 5 MitoPark mice (the legend's MP-early n = 5) and 7 controls, each mouse with 3 to 12
technical libraries. The supplementary files are this count table and an RPKM table of the same MitoPark libraries, and the series' design and its 92 samples describe MitoPark and control mice only, so there is no cMfn2KO table. The counts are integers, as DESeq2 and edgeR need.

## fig1b-anxa1-downregulated-rnaseq (MitoPark half)

> In bulk RNA-seq of sorted midbrain DANs, both early-stage mitochondrial PD models (cMfn2KO-early
> and MP-early) show significant downregulation of the Anxa1 transcript among DAN subtype markers.

Criterion: Anxa1 downregulated against its controls at FDR < 0.05 in both models. GEO holds no
differential expression results of the paper, so the readings are DESeq2 and edgeR with default
settings on the deposited counts, judged only where both agree. The deposit lacks the cMfn2KO-early
data, so that half cannot be tested and the claim stays blocked; the MitoPark half is run and
reported, as the answered question asks. The mouse is the unit: technical libraries are summed per
mouse (reading a); each library as a sample is a variant. All 5 MitoPark and 7 control mice are used.
DESeq2 is pydeseq2; edgeR is its Python port edgepy (TMM, common and tagwise dispersions, exact test),
since R is not in the entry's environment.

In [ ]:
by_mouse = GEO.T.groupby(GEO_LIBS.mouse.values).sum().T
rna = []
for unit, M in [("a mouse (libraries summed)", by_mouse), ("variant: library", GEO)]:
    g = np.array([c.split("_")[0] for c in M.columns])
    de, ed = anxa1.deseq2(M, g), anxa1.edger_exact(M, g)
    for tool, r, q in [("DESeq2", de, "padj"), ("edgeR", ed, "FDR")]:
        rna.append({"unit": unit, "tool": tool, "samples": f"{(g == 'MitoPark').sum()} MitoPark, {(g == 'Control').sum()} Control",
                    "Anxa1 log2FC": r.loc["Anxa1", "log2FoldChange"], "p": r.loc["Anxa1", "pvalue"], "FDR": r.loc["Anxa1", q],
                    "genes tested": int(r[q].notna().sum()), "genes at FDR < 0.05": int((r[q] < 0.05).sum()),
                    "Anxa1 down at FDR < 0.05": bool(r.loc["Anxa1", "log2FoldChange"] < 0 and r.loc["Anxa1", q] < 0.05)})
RNA = pd.DataFrame(rna)
print(RNA.to_string(index=False, formatters={"p": "{:.2e}".format, "FDR": "{:.2e}".format, "Anxa1 log2FC": "{:.3f}".format}))
cpm = by_mouse / by_mouse.sum() * 1e6
print("\nAnxa1 counts per million, per mouse:", cpm.loc["Anxa1"].round(1).to_dict())
print("cMfn2KO-early: no counts in GSE282753, so that half is not tested")

**Result: blocked (other).** The MitoPark half holds. With libraries summed per mouse (5 MitoPark,
7 controls), Anxa1 is down in MitoPark by DESeq2 (log2 fold change -1.02, FDR 2.9e-05) and by
edgeR (-0.99, FDR 7.5e-04), and both agree with each library as a sample (-0.93, FDR 5.0e-08 and
6.2e-08). Anxa1 counts per million are 90 to 222 in the MitoPark mice and 258 to 446 in the
controls. GSE282753 holds no cMfn2KO counts (its supplementary files are the MitoPark counts and RPKM only, and its 92 samples are MitoPark and control libraries), so the other half of the claim cannot be tested, and
the claim stays blocked.

## figs2-evoked-da-dcp-ccp

> Optogenetic activation of Anxa1+ DANs evokes DA release in the dCP at varying intensities, and the
> evoked release in the cCP has a lower amplitude.

Criterion: dLight z-scored signal aligned to the 1 s ChRmine trains; peak of the mean evoked
response within 0 to 1 s and within 0 to 2 s of onset; reproduced when the dCP peak exceeds the
pre-stimulus baseline and the cCP peak is lower than the dCP peak, per recording, by a two-sided
unpaired t-test at p < 0.05. After the answered question the criterion names the readings: (a) each
session's mean over all trains, all powers pooled, the session as unit (10 dCP against 10 cCP
sessions); (b) each power alone, the session as unit (2 against 2 at each of five powers). The
baseline is the mean over (i) 1 s and (ii) 2 s before train onset. The dandiset holds dF/F, not
z-scores, so the header's two z-scorings are run: over the session, and against the baseline
window before each train (the same 1 s or 2 s). A per-mouse summary is reported, not judged (two
mice per site). dCP and cCP are different mice, so no paired test applies.

In [ ]:
def evoked(s, zr, bl):
    x = anxa1.zscore(s["dff"]) if zr == "session" else s["dff"]
    g, P, kept = anxa1.peri(s["t"], x, s["on"], -2.0, 3.0)
    if zr == "baseline":
        pre = P[:, (g >= -bl) & (g < 0)]
        P = (P - pre.mean(1, keepdims=True)) / pre.std(1, keepdims=True)
    return g, P.mean(0), len(kept)

erows, traces = [], {}
for s in opto:
    for zr in ("session", "baseline"):
        for bl in (1.0, 2.0):
            g, m, n = evoked(s, zr, bl)
            traces[(s["subject"], float(s["power"][0]), zr, bl)] = (g, m)
            pre = m[(g >= -bl) & (g < 0)]
            for w in (1.0, 2.0):
                erows.append({"site": "dCP" if s["location"].startswith("Dorsal") else "cCP", "subject": s["subject"],
                              "power_mW": float(s["power"][0]), "zscore": zr, "baseline_s": bl, "window_s": w, "trains": n,
                              "trains_all": len(s["on"]), "peak": m[(g >= 0) & (g <= w)].max(), "baseline": pre.mean(), "pre_max": pre.max()})
ev = pd.DataFrame(erows)
ev["peak_minus_baseline"] = ev.peak - ev.baseline
per_sess = ev.drop_duplicates(["subject", "power_mW"])
print(f"Trains per session kept (window -2 to 3 s inside the recording): {per_sess.trains.min()} to {per_sess.trains.max()}; "
      f"dropped {int((per_sess.trains_all - per_sess.trains).sum())} of {int(per_sess.trains_all.sum())}")

def site_test(g):
    d, c = g[g.site == "dCP"], g[g.site == "cCP"]
    t = stats.ttest_ind(d.peak, c.peak)
    return {"n dCP": len(d), "n cCP": len(c), "dCP peak": d.peak.mean(), "cCP peak": c.peak.mean(),
            "dCP peak > baseline": f"{(d.peak > d.baseline).sum()}/{len(d)}",
            "dCP peak > pre-train max": f"{(d.peak > d.pre_max).sum()}/{len(d)}",
            "t": t.statistic, "p": t.pvalue, "cohen d": (d.peak.mean() - c.peak.mean()) / np.sqrt((d.peak.var() + c.peak.var()) / 2),
            "p (peak minus baseline)": stats.ttest_ind(d.peak_minus_baseline, c.peak_minus_baseline).pvalue,
            "holds": bool((d.peak > d.baseline).all() and d.peak.mean() > c.peak.mean() and t.pvalue < 0.05)}

KEYS = ["zscore", "baseline_s", "window_s"]
RA = pd.DataFrame([{**dict(zip(KEYS, k)), **site_test(g)} for k, g in ev.groupby(KEYS)])
print("Reading (a), all powers pooled per session, 10 sessions against 10:")
print(RA.round(4).to_string(index=False))
RB = pd.DataFrame([{**dict(zip(KEYS + ["power_mW"], k)), **site_test(g)} for k, g in ev.groupby(KEYS + ["power_mW"])])
print("\nReading (b), each power alone, 2 sessions against 2 (with n = 2 Cohen's d equals t):")
print(RB.drop(columns=["n dCP", "n cCP", "dCP peak > pre-train max"]).round(4).to_string(index=False))
print("\nReading (b), powers at which it holds, of 5:")
print(RB.groupby(KEYS).agg(powers_holding=("holds", "sum"), not_holding=("power_mW", lambda x: list(x[~RB.loc[x.index, "holds"]])),
                           p_min=("p", "min"), p_max=("p", "max"), d_min=("cohen d", "min"), d_max=("cohen d", "max")).round(4).to_string())

In [ ]:
main = ev[(ev.zscore == "session") & (ev.baseline_s == 1.0) & (ev.window_s == 1.0)]
print("Peak (session z-score, 0 to 1 s) by power and mouse:")
print(main.pivot_table(index="power_mW", columns=["site", "subject"], values="peak").round(3).to_string())
print("dCP above cCP in both mice at every power:",
      all(g[g.site == "dCP"].peak.min() > g[g.site == "cCP"].peak.max() for _, g in main.groupby("power_mW")))
print("Per mouse, mean over powers (reported, not judged):", main.groupby(["site", "subject"]).peak.mean().round(3).to_dict())
print("Spearman rho of peak with power, dCP:", round(stats.spearmanr(main[main.site == "dCP"].power_mW, main[main.site == "dCP"].peak)[0], 3),
      "| cCP:", round(stats.spearmanr(main[main.site == "cCP"].power_mW, main[main.site == "cCP"].peak)[0], 3))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, site in zip(axes, ["Dorsal", "Central"]):
    for s in [s for s in opto if s["location"].startswith(site)]:
        g, m = traces[(s["subject"], float(s["power"][0]), "session", 1.0)]
        ax.plot(g, m, color=plt.cm.viridis(np.log10(s["power"][0] / 0.1) / np.log10(20)), lw=1)
    ax.axvspan(0, 1, color="0.9"); ax.set_title(f"{site} caudoputamen"); ax.set_xlabel("time from train onset (s)")
axes[0].set_ylabel("dLight z (session)")
plt.tight_layout(); plt.show()
allm = np.concatenate([traces[k][1] for k in traces if k[2] == "session"])
print(f"plotted range: {allm.min():.2f} to {allm.max():.2f}; colour from 0.1 (dark) to 2 mW (light)")
print("\nMean session z-scored trace before train onset, all sessions of a site pooled (a train starts every 4 s,")
print("so -2 to -1 s is 2 to 3 s after the previous onset):")
for site in ("Dorsal", "Central"):
    ms = np.array([traces[(s["subject"], float(s["power"][0]), "session", 1.0)][1] for s in opto if s["location"].startswith(site)])
    gg = traces[next(iter(traces))][0]
    print(f"  {site}: -2 to -1 s {ms[:, (gg >= -2) & (gg < -1)].mean():.3f}, -1 to 0 s {ms[:, (gg >= -1) & (gg < 0)].mean():.3f}")


**Result: reproduced.** Under reading (a), all powers pooled per session, the criterion holds with
both z-scorings, both baselines and both windows: the dCP peak exceeds the baseline in 10 of 10
sessions and the cCP peak is lower (session z-score 1.21 against 0.61, t = 3.94, p = 0.0010,
Cohen's d 1.76; baseline z-score 2.30 against 1.19, p = 0.0041, with the 1 s baseline and 1.93
against 1.03, p = 0.0043, with 2 s). The 0 to 2 s window gives the same peaks as 0 to 1 s. Under
reading (b), each power alone with 2 sessions against 2, it holds at 3 or 4 of the 5 powers (p =
0.0039 to 0.037 where it holds); it fails at 2 mW under every combination (p = 0.16 to 0.31) and at
0.1 mW except with the baseline z-score over 1 s (p = 0.056 and 0.11 where it fails). The dCP peak
is above the cCP peak in both mice of each site at every power, and peaks rise with power (Spearman
rho 0.81 in dCP, 0.985 in cCP). Per mouse, reported and not judged, the mean peak over powers is
1.23 and 1.20 in dCP and 0.56 and 0.66 in cCP. The first reading the criterion lists holds in full,
so the outcome is reproduced; reading (b) holds at most powers but not at all. With trains every 4 s,
the 2 s baseline starts 2 s after the previous onset, but the mean trace is flat there (-0.175 from
-2 to -1 s and -0.174 from -1 to 0 s in dCP, -0.111 and -0.106 in cCP), so the previous response has
decayed and the 2 s baseline is not raised by it.

## fig3d-anxa1-ramp-to-touch

> Anxa1+ and DAT+ axons in dCP ramp up before a water drop, peak at contact with it, and return
> quickly to baseline during water consumption.

Criterion: z-scored signals aligned to the water drop touch; per recording the mean over touches of
the AUC in three windows, (a) 1 s windows -2 to -1, -1 to 0 and 0 to 1 s, (b) 2 s windows -4 to -2,
-2 to 0 and 0 to 2 s; RM one-way ANOVA with the Geisser-Greenhouse correction and Sidak's post hoc
(paired comparisons, Sidak over the three pairs), the recording as unit. Reproduced when, under a
reading, the Anxa1 ANOVA has p < 0.05 with window 2 above window 3 at Sidak p < 0.05, and the DAT
ANOVA has p < 0.05 with window 2 above window 1. The z-scoring readings of the header are run, (a)
over the session and (b) against a pre-event baseline, taken as the first window, which the legend
calls the baseline. After the answered questions the criterion names the touch: (i) `correct`
(drop retrieved) only, (ii) every contact with a drop (`correct`, `drop`, `semidrop`); the
other labels (`empty`, a reach to an empty spout, and `miss`) are used in neither. Every recording of the group and site
with at least one event is used, the n used is shown against the legend's, and a per-mouse summary
is reported, not judged.

In [ ]:
WINDOWS = {"1 s": [(-2, -1), (-1, 0), (0, 1)], "2 s": [(-4, -2), (-2, 0), (0, 2)]}
EVENTS = {"i correct": ["correct"], "ii contact": ["correct", "drop", "semidrop"]}

def drop_aucs(s, labels, windows, zr):
    e = s["events"][s["events"].label.isin(labels)].time.values
    x = anxa1.zscore(s["dff"]) if zr == "session" else s["dff"]
    g, P, kept = anxa1.peri(s["t"], x, e, windows[0][0], windows[2][1])
    if len(kept) == 0:
        return None, 0, len(e)
    if zr == "baseline":
        b = P[:, (g >= windows[0][0]) & (g <= windows[0][1])]
        P = (P - b.mean(1, keepdims=True)) / b.std(1, keepdims=True)
    return [float(anxa1.auc(g, P, *w).mean()) for w in windows], len(kept), len(e)

wrows = []
for s in [s for s in reach if s["fp"]]:
    for en, labels in EVENTS.items():
        for wn, ws in WINDOWS.items():
            for zr in ("session", "baseline"):
                a, n, n_all = drop_aucs(s, labels, ws, zr)
                wrows.append({"group": s["fp_group"], "subject": s["subject"], "start": s["start"][:16], "events": en,
                              "windows": wn, "zscore": zr, "n_events": n_all, "kept": n,
                              **({f"auc{i + 1}": a[i] for i in range(3)} if a else {})})
W = pd.DataFrame(wrows)
dropped = W[W.auc1.isna()].drop_duplicates(["subject", "start", "events"])
print("Recordings without any usable event, by event set:", dropped.groupby("events").size().to_dict())
lost = W[W.zscore == "session"].assign(lost=lambda d: d.n_events - d.kept.fillna(0)).groupby(["events", "windows"]).agg(events=("n_events", "sum"), lost=("lost", "sum"))
print("Events, and events whose window left the recording:"); print(lost.to_string())
W = W.dropna(subset=["auc1"])

def window_tests(g):
    v = g[["auc1", "auc2", "auc3"]].values
    r = anxa1.rm_oneway(v)
    ph = anxa1.paired_sidak(v, [(0, 1), (0, 2), (1, 2)])
    return {"n": len(v), "mice": g.subject.nunique(), "mean auc1": v[:, 0].mean(), "mean auc2": v[:, 1].mean(), "mean auc3": v[:, 2].mean(),
            "F": r["F"], "df": f"{r['df1']:.2f}, {r['df2']:.2f}", "p": r["p"],
            "Sidak 1v2": ph["p Sidak"][0], "Sidak 1v3": ph["p Sidak"][1], "Sidak 2v3": ph["p Sidak"][2]}

TESTS = pd.DataFrame([{"group": k[0], "events": k[1], "windows": k[2], "zscore": k[3], **window_tests(g)}
                      for k, g in W.groupby(["group", "events", "windows", "zscore"])])
def show(group):
    print(TESTS[TESTS.group == group].drop(columns="group").round(4).to_string(index=False))
for grp in ["Anxa1 GCaMP-dCP", "DAT GCaMP-dCP"]:
    print("\n" + grp); show(grp)

In [ ]:
T = TESTS.set_index(["group", "events", "windows", "zscore"])
rows = []
for en in EVENTS:
    for wn in WINDOWS:
        for zr in ("session", "baseline"):
            a, d = T.loc[("Anxa1 GCaMP-dCP", en, wn, zr)], T.loc[("DAT GCaMP-dCP", en, wn, zr)]
            anxa = a.p < 0.05 and a["mean auc2"] > a["mean auc3"] and a["Sidak 2v3"] < 0.05
            dat = d.p < 0.05 and d["mean auc2"] > d["mean auc1"] and d["Sidak 1v2"] < 0.05
            rows.append({"events": en, "windows": wn, "zscore": zr, "Anxa1 part holds": anxa, "DAT part holds": dat, "both": anxa and dat})
print(pd.DataFrame(rows).to_string(index=False))
per_mouse = W[(W.events == "i correct") & (W.windows == "1 s") & (W.zscore == "session")].groupby(["group", "subject"])[["auc1", "auc2", "auc3"]].mean()
print("\nPer mouse (events i, 1 s windows, session z), mean AUC per window:")
print(per_mouse.loc[["Anxa1 GCaMP-dCP", "DAT GCaMP-dCP"]].round(3).to_string())
print("\nDAT GCaMP-dCP without mouse 687662 (anxa1-flp in its open field file, Dat_cre here):")
print(pd.DataFrame([{"events": k[0], "windows": k[1], "zscore": k[2], **window_tests(g)}
                    for k, g in W[(W.group == "DAT GCaMP-dCP") & (W.subject != "687662")].groupby(["events", "windows", "zscore"])])
      [["events", "windows", "zscore", "n", "mice", "p", "Sidak 1v2"]].round(4).to_string(index=False))


In [ ]:
LEGEND_N = {"Anxa1 GCaMP-dCP": (10, 7), "DAT GCaMP-dCP": (4, 3), "dLight-dCP": (13, 6), "dLight-cCP": (11, 6)}
nused = TESTS[TESTS.zscore == "session"].pivot_table(index="group", columns=["events", "windows"], values=["n", "mice"]).astype(int)
nused[("legend", "recordings", "")] = [LEGEND_N[g][0] for g in nused.index]
nused[("legend", "mice", "")] = [LEGEND_N[g][1] for g in nused.index]
print("Recordings (n) and mice used, under each event and window reading, against the legend of 3D and 3E:")
print(nused.to_string())

def mouse_tests(g):
    v = g.groupby("subject")[["auc1", "auc2", "auc3"]].mean().values
    r = anxa1.rm_oneway(v)
    ph = anxa1.paired_sidak(v, [(0, 1), (0, 2), (1, 2)])
    return {"mice": len(v), "mean auc1": v[:, 0].mean(), "mean auc2": v[:, 1].mean(), "mean auc3": v[:, 2].mean(), "p": r["p"],
            "Sidak 1v2": ph["p Sidak"][0], "Sidak 1v3": ph["p Sidak"][1], "Sidak 2v3": ph["p Sidak"][2]}
MOUSE = pd.DataFrame([{"group": k[0], "events": k[1], "windows": k[2], "zscore": k[3], **mouse_tests(g)}
                      for k, g in W.groupby(["group", "events", "windows", "zscore"])])
print("\nPer-mouse summary (each mouse's mean over its recordings; RM ANOVA and Sidak over mice), reported, not judged:")
print(MOUSE.round(4).to_string(index=False))

In [ ]:
def mean_traces(group, labels, lo=-5, hi=5):
    out = []
    for s in [s for s in reach if s["fp"] and s["fp_group"] == group]:
        e = s["events"][s["events"].label.isin(labels)].time.values
        g, P, kept = anxa1.peri(s["t"], anxa1.zscore(s["dff"]), e, lo, hi)
        if len(kept):
            out.append(P.mean(0))
    return g, np.array(out)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharey=True)
rng_all = []
for ax, grp in zip(axes, anxa1.FP_GROUPS):
    g, M = mean_traces(grp, ["correct"])
    m, se = M.mean(0), M.std(0) / np.sqrt(len(M))
    rng_all += [m.min(), m.max()]
    ax.plot(g, m, color="k"); ax.fill_between(g, m - se, m + se, color="0.8")
    for x in (-2, -1, 0, 1): ax.axvline(x, color="0.85", lw=0.5)
    ax.set_title(f"{grp} (n={len(M)})"); ax.set_xlabel("time from drop retrieval (s)")
    print(f"{grp}: mean trace peaks at {g[np.argmax(m)]:+.2f} s, value {m.max():.2f}; at 0 s {m[np.argmin(np.abs(g))]:.2f}")
axes[0].set_ylabel("z (session)"); plt.tight_layout(); plt.show()
print(f"plotted means range {min(rng_all):.2f} to {max(rng_all):.2f}")

**Result: not reproduced.** Under no reading do both parts hold. The DAT part fails in all eight
combinations of events, windows and z-scoring (ANOVA p = 0.050 to 0.145, window 1 against 2 Sidak
p = 0.35 to 0.9985). The Anxa1 part holds only with every contact as the touch, 1 s windows and the
session z-score (ANOVA p = 0.048, window 2 above window 3 at Sidak p = 0.045); with the drop
retrievals alone its ANOVA has p = 0.0139 to 0.109 and window 2 against 3 Sidak p = 0.107 to 0.211.
The mean Anxa1 trace does peak at +0.03 s from the drop retrieval (z 0.25), the DAT trace at +2.88
s. The answered questions fixed the event readings and the recordings as they were already run, so
the numbers and the outcome are unchanged. Every recording has at least one event under both event
readings: 15 Anxa1 recordings of 6 mice and 8 DAT of 5, against the legend's 10 of 7 and 4 of 3; 3
to 5 of 661 and 731 events are lost where the window leaves the recording. Per mouse, reported and
not judged, the DAT ANOVA has p = 0.10 to 0.25, the Anxa1 ANOVA p = 0.027 to 0.15 and its window 2
against 3 Sidak p = 0.032 to 0.16. Mouse 687662 is anxa1-flp in its open field file and Dat_cre
here; without it the DAT part still fails in every reading (ANOVA p = 0.055 to 0.143, window 1
against 2 Sidak p = 0.46 to 0.99).

## fig3e-dlight-dcp-ccp-reward

> DA release in dCP follows the same ramp-and-drop dynamics as Anxa1+ axons, while in cCP it shows a
> classical reward-related profile, ramping before contact and peaking during consumption before
> returning to baseline.

Criterion: as fig3d, with the same window, event and recording readings (the event and recording
readings fixed by the answered questions). Reproduced when, under a reading, dLight-dCP has ANOVA p < 0.05
with window 2 above window 3 at Sidak p < 0.05, and dLight-cCP has ANOVA p < 0.05 with windows 2
and 3 both above window 1 at Sidak p < 0.05. The test tables are those computed above.

In [ ]:
for grp in ["dLight-dCP", "dLight-cCP"]:
    print("\n" + grp); show(grp)
rows = []
for en in EVENTS:
    for wn in WINDOWS:
        for zr in ("session", "baseline"):
            d, c = T.loc[("dLight-dCP", en, wn, zr)], T.loc[("dLight-cCP", en, wn, zr)]
            dcp = d.p < 0.05 and d["mean auc2"] > d["mean auc3"] and d["Sidak 2v3"] < 0.05
            ccp = (c.p < 0.05 and c["mean auc2"] > c["mean auc1"] and c["mean auc3"] > c["mean auc1"]
                   and c["Sidak 1v2"] < 0.05 and c["Sidak 1v3"] < 0.05)
            rows.append({"events": en, "windows": wn, "zscore": zr, "dCP part holds": dcp, "cCP part holds": ccp, "both": dcp and ccp})
print(); print(pd.DataFrame(rows).to_string(index=False))
pm = W[(W.events == "i correct") & (W.windows == "1 s") & (W.zscore == "session")].groupby(["group", "subject"])[["auc1", "auc2", "auc3"]].mean()
print("\nPer mouse (events i, 1 s, session z):"); print(pm.loc[["dLight-dCP", "dLight-cCP"]].round(3).to_string())

**Result: not reproduced.** The dCP part, the point of the claim (DA in dCP follows the Anxa1
ramp and drop), does not hold under any reading: its ANOVA has p = 0.21 to 0.76 and window 2 against
3 Sidak p = 0.30 to 0.90. The cCP profile is there: its ANOVA has p = 0.0001 to 0.0014 under every
reading, window 3 is above window 1 at Sidak p = 0.0005 to 0.0012, and window 2 is above window 1 at
Sidak p < 0.05 in 5 of 8 combinations (for example drop retrievals, 1 s windows, session z: F =
13.42, p = 0.0006; 1 against 2 Sidak p = 0.041; 1 against 3 p = 0.0006), so the cCP part holds in 5
of 8. The criterion needs both parts under one reading, which never happens. This was partially
reproduced before the first verification; the verifier judged it not reproduced, since the dCP part
fails everywhere, and the outcome follows that. The dandiset gives 21 dCP recordings of 7 mice and 15
cCP of 6, against the legend's 13 of 6 and 11 of 6. Per mouse, reported and not judged, the dCP
ANOVA has p = 0.46 to 0.87 and the cCP ANOVA p = 0.0006 to 0.018.


## fig3fg-auc-change-from-baseline

> Against the baseline window, the legend lists three AUC changes, the second window of DAT
> GCaMP-dCP (p = 0.0308) and of dLight-cCP (p = 0.0028) and the third window of dLight-cCP (p =
> 0.0148), with only the dLight-cCP second window marked significant; the other groups and windows
> are not listed.

Criterion: per recording the AUC of windows 2 and 3 minus window 1; one-sample two-tailed t-test
against 0, Bonferroni over eight tests (four groups, two windows) or over four (per window).
Reproduced when, under a window reading, the dLight-cCP second-window change is positive and
significant after Bonferroni under a family, and the other two listed changes are positive. The
claims file names both z-scorings and ranks neither, so both are judged. Under the baseline z-score
the first window is the baseline the trace is z-scored against.

In [ ]:
crow = []
for (en, wn, zr), g0 in W.groupby(["events", "windows", "zscore"]):
    for grp in anxa1.FP_GROUPS:
        g = g0[g0.group == grp]
        for w in (2, 3):
            dlt = g[f"auc{w}"] - g.auc1
            t = stats.ttest_1samp(dlt, 0)
            crow.append({"events": en, "windows": wn, "zscore": zr, "group": grp, "window": w, "n": len(dlt),
                         "mean change": dlt.mean(), "p raw": t.pvalue})
C = pd.DataFrame(crow)
C["p Bonferroni x8"] = (C["p raw"] * 8).clip(upper=1)
C["p Bonferroni x4"] = (C["p raw"] * 4).clip(upper=1)
for zr in ("session", "baseline"):
    print(f"z-score over the {zr}:" if zr == "session" else "z-score against the baseline window:")
    print(C[C.zscore == zr].drop(columns="zscore").round(4).to_string(index=False))
rows = []
for (en, wn, zr), g in C.groupby(["events", "windows", "zscore"]):
    g = g.set_index(["group", "window"])
    c2 = g.loc[("dLight-cCP", 2)]
    others = g.loc[("DAT GCaMP-dCP", 2), "mean change"] > 0 and g.loc[("dLight-cCP", 3), "mean change"] > 0
    rows.append({"events": en, "windows": wn, "zscore": zr, "cCP 2nd change": c2["mean change"],
                 "x8": c2["p Bonferroni x8"], "x4": c2["p Bonferroni x4"],
                 "holds x8": c2["mean change"] > 0 and c2["p Bonferroni x8"] < 0.05 and others,
                 "holds x4": c2["mean change"] > 0 and c2["p Bonferroni x4"] < 0.05 and others,
                 "DAT 2nd change": g.loc[("DAT GCaMP-dCP", 2), "mean change"], "cCP 3rd change": g.loc[("dLight-cCP", 3), "mean change"]})
print(); print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: partially reproduced.** The claims file names two z-scorings and judges the claim under
both; it holds under one and fails under the other. With the baseline z-score the dLight-cCP
second-window change is positive and significant after Bonferroni over four tests in all four event
and window combinations (p = 0.020 to 0.049) and over eight with 2 s windows (p = 0.040 and 0.047),
with the DAT second (0.13 to 0.28) and cCP third (1.43 to 1.77) changes positive. With the session
z-score the cCP second-window change (0.29 to 0.36) is not significant after either correction (four
tests p = 0.055 to 0.31). In every reading the dLight-cCP third window departs from the first far
more than the second does (raw p = 0.0002 to 0.0004 against 0.005 to 0.077), the reverse of the
legend, where the second (p = 0.0028) is stronger than the third (p = 0.0148) and only the second is
marked. The DAT second-window change is positive but far from significant (raw p = 0.13 to 0.89).
This was blocked as methods underspecified before the first verification, on the view that the
z-scoring decides; the verifier judged it partially reproduced, as it holds under one of the claims
file's readings only, and the outcome follows that.


## fig4b-speed-correlation-dcp

> Anxa1+ axonal activity in the dCP correlates positively with linear speed during open field
> exploration, above a shuffled control, and DA release in dCP shows a similar correlation.

Criterion: per recording, Pearson's r over the session between the z-scored photometry and linear
speed, against the same with the signal shuffled, (a) a random permutation of samples and (b) a
circular shift by a random lag; two-sided paired t-test over recordings; reproduced when, under a
reading, both Anxa1 GCaMP-dCP and dLight-dCP have a mean r above 0 and above the shuffle at p <
0.05. The dandiset's 30 Hz z-scored trace (the paper's time base) and its `linear_velocity` are
used. The shuffled value of a recording is the mean over 100 shuffles (a standard choice that cannot
change the outcome, as shuffled r is near 0). The Wilcoxon signed-rank test is a variant.

In [ ]:
rng_4b = np.random.default_rng(4)
crows = []
for s in fp:
    z, v, _ = anxa1.frame_aligned(s)
    ok = np.isfinite(z) & np.isfinite(v)
    z, v = z[ok], v[ok]
    n = len(z)
    crows.append({"group": s["group"], "subject": s["subject"], "start": s["start"][:16], "samples": n,
                  "r": np.corrcoef(z, v)[0, 1],
                  "r permuted": np.mean([np.corrcoef(rng_4b.permutation(z), v)[0, 1] for _ in range(100)]),
                  "r circular": np.mean([np.corrcoef(np.roll(z, rng_4b.integers(1, n)), v)[0, 1] for _ in range(100)])})
CR = pd.DataFrame(crows).sort_values(["group", "subject", "start"])
print(CR.round(4).to_string(index=False))
def corr_tests(g):
    out = {"recordings": len(g), "mice": g.subject.nunique(), "mean r": g.r.mean(), "r > 0": f"{(g.r > 0).sum()}/{len(g)}"}
    for sh in ("permuted", "circular"):
        out[f"t-test p vs {sh}"] = stats.ttest_rel(g.r, g[f"r {sh}"]).pvalue
        out[f"Wilcoxon p vs {sh}"] = stats.wilcoxon(g.r, g[f"r {sh}"]).pvalue
    out["mean r per mouse"] = g.groupby("subject").r.mean().round(3).to_dict()
    return out
CT = pd.DataFrame([{"group": k, **corr_tests(g)} for k, g in CR.groupby("group")])
print(); print(CT.drop(columns="mean r per mouse").round(4).to_string(index=False))
for _, r in CT.iterrows(): print(r.group, "per mouse:", r["mean r per mouse"])
print("\nWithout the seven recordings holding under 13 min of photometry:")
long = CR[CR.samples > 13 * 60 * 30]
print(pd.DataFrame([{"group": k, **corr_tests(g)} for k, g in long.groupby("group")]).drop(columns="mean r per mouse").round(4).to_string(index=False))
print("\nAnxa1 GCaMP-dCP without mouse 687662 (Dat_cre in its water drop file):")
print(pd.DataFrame([corr_tests(CR[(CR.group == "Anxa1 GCaMP-dCP") & (CR.subject != "687662")])]).drop(columns="mean r per mouse").round(4).to_string(index=False))
print("\nWith each recording's speed moved onto the pose frames by its kinematics lag (data check above;"
      " recordings at lag 0 unchanged):")
rng_lag = np.random.default_rng(4)
lrows = []
for s in fp:
    l = int(KLAG[s["path"]])
    z, v, _ = anxa1.frame_aligned({**s, "lin": np.r_[np.full(l, np.nan), s["lin"]][:len(s["lin"])]})
    ok = np.isfinite(z) & np.isfinite(v)
    z, v = z[ok], v[ok]
    n = len(z)
    lrows.append({"group": s["group"], "subject": s["subject"], "lag": l, "r": np.corrcoef(z, v)[0, 1],
                  "r permuted": np.mean([np.corrcoef(rng_lag.permutation(z), v)[0, 1] for _ in range(100)]),
                  "r circular": np.mean([np.corrcoef(np.roll(z, rng_lag.integers(1, n)), v)[0, 1] for _ in range(100)])})
LR = pd.DataFrame(lrows)
print("Recordings moved:", LR[LR.lag != 0][["group", "subject", "lag"]].values.tolist())
print(pd.DataFrame([{"group": k, **corr_tests(g)} for k, g in LR.groupby("group")]).drop(columns="mean r per mouse").round(4).to_string(index=False))
print("\nLag of the peak speed-signal correlation per Anxa1 GCaMP-dCP recording (data check above), with its r here:")
print(fpdf[fpdf.group == "Anxa1 GCaMP-dCP"][["subject", "start", "lag_of_peak_|r|_s", "peak_|r|"]]
      .merge(CR[["subject", "start", "r"]], on=["subject", "start"]).round(3).to_string(index=False))


**Result: not reproduced.**
dLight-dCP correlates with speed above both shuffles (mean r = 0.067, 7 of 8 recordings positive,
paired t-test p = 0.010 against permutation and 0.008 against circular shifts; Wilcoxon p = 0.023
and 0.016). Anxa1 GCaMP-dCP is positive in 9 of 10 recordings but not significantly (mean r = 0.066,
p = 0.080 and 0.071, Wilcoxon p = 0.064); one recording (842430) is at r = -0.17. Without the seven
short recordings, Anxa1 p = 0.097 and 0.089, and dLight-dCP p = 0.059 against permutation and 0.0499
against circular shifts. DAT GCaMP-dCP, with no test in the legend, has mean r = 0.032 (p = 0.22).
Without mouse 687662, whose water drop file says Dat_cre, Anxa1 has mean r = 0.057, 7 of 8 positive,
p = 0.21 and 0.18.

Of the 19 open field files whose kinematics are not at lag 0 to the pose, five hold photometry
(Anxa1 687642 at 5 frames and 816510 at 1, dLight-dCP 776769 at 1 and 829248 at 30, dLight-cCP
842388 at 1). With their speed moved by that lag, Anxa1 p = 0.082 and 0.073 and dLight-dCP p = 0.028
and 0.023 (6 of 8
positive, Wilcoxon p = 0.0547 against permutation), so the result does not change. The speed-signal correlation of the Anxa1 recordings peaks
within 0.5 s of zero lag in 5 of 10; the other five peak at -2.3 to 2.1 s, on both sides of zero, with
peak |r| 0.09 to 0.19. That the photometry and the speed share one clock is checked where the files
allow it, not shown for every recording, so an offset in some recordings is not ruled out.

The criterion needs both groups at p < 0.05 under one reading, and Anxa1, the subject of the claim,
misses it in every reading (p = 0.064 to 0.21). As for fig3e, whose main part fails in every reading,
this is a failure of the claim and not a partial result. This was partially reproduced before the
first verification; the verifier judged it not reproduced in rounds 1 and 2, the change was proposed
after round 2, and the third round agreed with it, so the outcome follows that.


## fig4b-speed-correlation-ccp-negative

> DA release in the cCP correlates negatively with locomotion speed.

Criterion: as above; reproduced when the mean dLight-cCP correlation is below 0 and below the
shuffle at p < 0.05 under a reading (Wilcoxon as a variant). The values are in the tables above.

In [ ]:
print(CT[CT.group == "dLight-cCP"].drop(columns="mean r per mouse").round(5).to_string(index=False))
print("per recording r:", CR[CR.group == "dLight-cCP"].r.round(3).tolist())

**Result: reproduced.** dLight-cCP correlates negatively with speed in all 7 recordings (mean r =
-0.158, from -0.26 to -0.07), below the permutation and circular shuffles (paired t-test p = 0.00045
and 0.00054; Wilcoxon p = 0.0156, the exact 2/128 the claims file notes).

## fig4c-vame-motifs

> Unsupervised VAME analysis of open field pose identifies 37 kinematic motifs, grouped into action
> categories.

Criterion: the count of distinct motif labels in the paper's motif labelling; tolerance 0. The
dandiset holds one labelling, `VAME_42/MotifSeries`, in every open field file; whether it is the
paper's 37-motif labelling is checked against the motif numbers the Results name. The claims file's
header says the motif claims use the paper's label per frame if the dataset holds it, and that a new
VAME fit is not a basis for an outcome.

In [ ]:
counts = np.zeros(60, int)
for s in oft:
    counts += np.bincount(s["motif"], minlength=60)
used = np.flatnonzero(counts)
usage = counts / counts.sum()
print("Distinct labels over all", len(oft), "open field files:", len(used), "| labels", used.min(), "to", used.max())
print("Labels with at least 1% of frames (the Methods' usage threshold):", int((usage >= 0.01).sum()))
f5 = np.zeros(60, int)
for s in oft:
    if s["group"] in anxa1.FIG5_GROUPS:
        f5 += np.bincount(s["motif"], minlength=60)
print("Labels never used by the 108 mice of Figure 5:", np.flatnonzero((f5 == 0) & (counts > 0)).tolist())
named = {"42": "reduced in TeTx and all models", "34": "reduced in TeTx and all models", "16": "reduced in TeTx",
         "0": "reduced in TeTx and MP-middle", "25": "reduced in TeTx and MP-middle",
         "3": "stationary, up in MP-middle", "39": "stationary, up in MP-middle", "27": "stationary, up in MP-middle",
         "28": "stationary, up in MP-middle"}       # motif numbers named in the Results, Figure 5J and 5M
print("Motifs named by the Results and their use in the Figure 5 mice:")
for k, why in named.items():
    k = int(k)
    print(f"  motif {k:2d} ({why}): {'not a label of the dandiset' if k > used.max() else f'{f5[k] / f5.sum() * 100:.2f}% of frames'}")
print("Every label's share of frames over all files (%):", np.round(usage[used] * 100, 2).tolist())

**Result: blocked (data missing).** The dandiset's VAME labelling has 42 distinct labels (0 to 41),
39 of them above 1% of frames, where the paper reports 37 motifs (47 above the 1% threshold, 10 then
merged, by the Methods). It is not the paper's labelling: motif 42, which the Results name, is not a
label of the dandiset, and motif 28, named as a stationary motif increased in MP-middle mice, is never
used by the 108 mice of Figure 5. No motif category is stored. The paper's labelling is therefore not
in the dandiset, and by the claims file's header this claim and those resting on it (fig4e, fig4f,
fig4h, fig5g, fig5h, fig5i, fig5j and fig5m) have no basis for an outcome. Before the first
verification this claim was judged not reproduced (42 labels against 37) and fig4e, fig4f, fig4h,
fig5g and fig5i were judged on the dandiset's labels; the verifier pointed out that this contradicts
the header and the blocking of fig5j and fig5m on the same evidence. The analyses below are kept, on
the dandiset's labels, and decide nothing.


## fig4f-auc-correlations

> Across motifs, the onset AUC of Anxa1+ terminal activity correlates positively with that of
> dLight in dCP (R = 0.70) and negatively with that of dLight in cCP (R = -0.52).

Criterion: per motif, the AUC from 0 to 1 s after onset (first frame of a run of the label) of the
z-scored signal, then Pearson's R over motifs between groups. Readings: (a) all onsets of a group
pooled, then the mean trace, or (b) the mean per recording, then the mean over recordings; (i) the
AUC as is, or (ii) minus the mean over -1 to 0 s. Reproduced when, under a reading, both R are
within tolerance (0.70 +/- 0.07, -0.52 +/- 0.052) and p < 0.01. The dandiset's labels are used
(see fig4c), as stored; a motif enters when every group has an onset of it with 1 s of signal on both sides.

Blocked (data missing), as it rests on the paper's motif labelling (fig4c). The test below runs on
the dandiset's own labels and is shown, not judged.


In [ ]:
NM = 60
grid = np.arange(-30, 31) / 30.0

def onset_aucs(shift=0):
    """Mean onset traces and 0 to 1 s AUCs per motif and group, the labels shifted by `shift` frames."""
    onset_traces = {grp: {k: [] for k in range(NM)} for grp in anxa1.FP_GROUPS}
    per_rec = {grp: [] for grp in anxa1.FP_GROUPS}
    n_on = {grp: np.zeros(NM, int) for grp in anxa1.FP_GROUPS}
    for s in fp:
        z, v, m = anxa1.frame_aligned(s, shift)
        rec = np.full((NM, len(grid)), np.nan)
        for k, idx in anxa1.motif_onsets(m).items():
            idx = idx[(idx >= 30) & (idx + 30 < len(z))]
            if len(idx) == 0: continue
            tr = np.stack([z[i - 30:i + 31] for i in idx])
            tr = tr[np.isfinite(tr).all(1)]
            if len(tr) == 0: continue
            rec[k] = tr.mean(0); onset_traces[s["group"]][k].append(tr); n_on[s["group"]][k] += len(tr)
        per_rec[s["group"]].append(rec)
    out = {}
    for grp in anxa1.FP_GROUPS:
        for base in (False, True):
            out[(grp, "a pooled", base)] = np.array([motif_auc(np.concatenate(onset_traces[grp][k]).mean(0), base)
                                                     if onset_traces[grp][k] else np.nan for k in range(NM)])
            out[(grp, "b per recording", base)] = np.nanmean(np.stack([motif_auc(r, base) for r in per_rec[grp]]), 0)
    return out, n_on

def motif_auc(trace, base):
    a = anxa1.auc(grid, trace, 0, 1)
    return a - trace[..., (grid >= -1) & (grid < 0)].mean(-1) * 1.0 if base else a

MAUC, n_onsets = onset_aucs(0)
common = np.flatnonzero(np.all([n_onsets[g] > 0 for g in ["Anxa1 GCaMP-dCP", "dLight-dCP", "dLight-cCP"]], axis=0))
print("Motifs with onsets in Anxa1, dLight-dCP and dLight-cCP:", len(common), "of", len(used),
      "| missing:", sorted(set(used) - set(common)))
print("Onsets per motif, Anxa1: median", int(np.median(n_onsets["Anxa1 GCaMP-dCP"][common])), "min", n_onsets["Anxa1 GCaMP-dCP"][common].min())
rows = []
for rd in ("a pooled", "b per recording"):
    for base in (False, True):
        a, d, c = (MAUC[(g, rd, base)][common] for g in ["Anxa1 GCaMP-dCP", "dLight-dCP", "dLight-cCP"])
        r1, r2 = stats.pearsonr(a, d), stats.pearsonr(a, c)
        rows.append({"averaging": rd, "baseline subtracted": base, "R Anxa1-dLight dCP": r1[0], "p": r1[1],
                     "R Anxa1-dLight cCP": r2[0], "p ": r2[1],
                     "within tolerance": abs(r1[0] - 0.70) <= 0.07 and abs(r2[0] + 0.52) <= 0.052 and r1[1] < 0.01 and r2[1] < 0.01})
F4 = pd.DataFrame(rows); print(F4.round(4).to_string(index=False))


**Result: blocked (data missing).** On the dandiset's labels as stored, reading (b)(ii), the mean per
recording with the -1 to 0 s mean subtracted, gives R = 0.692 between Anxa1 and dLight-dCP and R =
-0.571 between Anxa1 and dLight-cCP (both p < 0.0001), each within its tolerance; reading (a)(ii)
gives 0.676 and -0.590 (the second 0.018 outside), and without baseline subtraction R is 0.44 to 0.46
and -0.34 to -0.40. These are over 39 dandiset labels, not the paper's 37 motifs, and the labels are
early by about 18 frames (data check). With the labels moved, no reading is within both tolerances
(table under fig4h): at +18 frames R is 0.39 to 0.69 and -0.19 to -0.39, at +15 0.39 to 0.68 and
-0.23 to -0.38, at -15 0.69 and -0.75 or -0.40 with baseline subtraction.


## fig4h-auc-motif-speed

> The onset AUC of Anxa1+ terminal activity is highly correlated with mean motif linear speed across
> motifs (R = 0.83).

Criterion: the AUC readings of fig4f; motif speed is the mean linear speed over the frames of the
motif, over the Anxa1 recordings or over all mice of the motif set (all open field files), both run.
Reproduced when R is within 0.83 +/- 0.083 and p < 0.01 under a reading.

Blocked (data missing), as it rests on the paper's motif labelling (fig4c). The test below runs on
the dandiset's own labels and is shown, not judged.


In [ ]:
def motif_speed(sessions, shift=0):
    tot, n = np.zeros(NM), np.zeros(NM)
    for s in sessions:
        m = anxa1.shift_labels(s["motif"], shift)
        v = s["lin"][:len(m)]
        ok = np.isfinite(v) & (m >= 0)
        tot += np.bincount(m[ok], weights=v[ok], minlength=NM); n += np.bincount(m[ok], minlength=NM)
    return tot / np.maximum(n, 1)
SPEED = {"Anxa1 recordings": motif_speed([s for s in fp if s["group"] == "Anxa1 GCaMP-dCP"]), "all open field files": motif_speed(oft)}
rows = []
for rd in ("a pooled", "b per recording"):
    for base in (False, True):
        for sp, spd in SPEED.items():
            r = stats.pearsonr(MAUC[("Anxa1 GCaMP-dCP", rd, base)][common], spd[common])
            rows.append({"averaging": rd, "baseline subtracted": base, "speed over": sp, "R": r[0], "p": r[1],
                         "within tolerance": abs(r[0] - 0.83) <= 0.083 and r[1] < 0.01})
H4 = pd.DataFrame(rows); print(H4.round(5).to_string(index=False))
fig, ax = plt.subplots(figsize=(4, 3.5))
x, y = SPEED["all open field files"][common], MAUC[("Anxa1 GCaMP-dCP", "b per recording", True)][common]
ax.scatter(x, y, s=12); ax.set_xlabel("mean motif speed (cm/s)"); ax.set_ylabel("Anxa1 onset AUC, reading b(ii)")
plt.tight_layout(); plt.show()
print(f"plotted: speed {x.min():.2f} to {x.max():.2f} cm/s, AUC {y.min():.3f} to {y.max():.3f}")
print("\nLabel timing check (data check): the readings above with the labels moved by a number of frames")
print(f"(+ is later; {BEST_SHIFT:+d} is the median shift at which the labels explain speed best):")
srows = []
for sh in sorted({-30, -15, 0, 15, 30, BEST_SHIFT}):
    M_ = MAUC if sh == 0 else onset_aucs(sh)[0]
    sp = motif_speed(oft, sh)[common]
    for rd in ("a pooled", "b per recording"):
        for base in (False, True):
            a, d, c = (M_[(g, rd, base)][common] for g in ["Anxa1 GCaMP-dCP", "dLight-dCP", "dLight-cCP"])
            srows.append({"shift (frames)": sh, "averaging": rd, "baseline subtracted": base,
                          "R Anxa1-dLight dCP": stats.pearsonr(a, d)[0], "R Anxa1-dLight cCP": stats.pearsonr(a, c)[0],
                          "R Anxa1-speed (all files)": stats.pearsonr(a, sp)[0],
                          "4f within tolerance": abs(stats.pearsonr(a, d)[0] - 0.70) <= 0.07 and abs(stats.pearsonr(a, c)[0] + 0.52) <= 0.052,
                          "4h within tolerance": abs(stats.pearsonr(a, sp)[0] - 0.83) <= 0.083})
SH = pd.DataFrame(srows); print(SH.round(3).to_string(index=False))


**Result: blocked (data missing).** On the dandiset's labels as stored, Anxa1 onset AUC correlates
positively with mean motif speed under every reading, but below the paper's value: R = 0.62 to 0.68
with baseline subtraction (p < 0.0001) and 0.47 to 0.55 without (p = 0.0003 to 0.0025), none within
0.83 +/- 0.083. The result depends on where the labels sit on the frame grid. With them moved to
where they explain speed best (+18 frames), R falls to 0.27 to 0.31; moved 15 frames earlier, one
reading reaches 0.781, within the tolerance; moved 30 frames either way, R is -0.42 to 0.14. The
motif-onset AUCs on these labels therefore say little about the claim.


## fig4e-anxa1-high-speed-motifs

> Anxa1+ axon activity is associated with the onset of high-speed motifs regardless of action type,
> as is DA in dCP, while DA in cCP rises strongly after the onset of low-speed motifs.

Criterion: judged by fig4f and fig4h, and in addition, for "independently of the action type", a
positive Spearman correlation of Anxa1 AUC (0 to 1 s) with motif speed within at least two action
categories, and for cCP a negative correlation of its AUC with motif speed. The dandiset holds no
category for its motifs, so the within-category part cannot be run. The other parts are below.

Blocked (data missing), as it rests on the paper's motif labelling (fig4c). The test below runs on
the dandiset's own labels and is shown, not judged.


In [ ]:
rows = []
for rd in ("a pooled", "b per recording"):
    for base in (False, True):
        for grp in ["Anxa1 GCaMP-dCP", "dLight-dCP", "dLight-cCP"]:
            spd = SPEED["all open field files"][common]
            rho = stats.spearmanr(MAUC[(grp, rd, base)][common], spd)
            r = stats.pearsonr(MAUC[(grp, rd, base)][common], spd)
            rows.append({"averaging": rd, "baseline subtracted": base, "group": grp, "Spearman rho": rho[0], "p": rho[1], "Pearson R": r[0], "p ": r[1]})
print(pd.DataFrame(rows).round(5).to_string(index=False))
q = SPEED["all open field files"][common]
slow, fastm = common[q <= np.percentile(q, 25)], common[q >= np.percentile(q, 75)]
for grp in ["Anxa1 GCaMP-dCP", "dLight-dCP", "dLight-cCP"]:
    a = MAUC[(grp, "b per recording", True)]
    print(f"{grp}: mean onset AUC b(ii), slowest quarter of motifs {np.nanmean(a[slow]):+.3f}, fastest quarter {np.nanmean(a[fastm]):+.3f}")

**Result: blocked (data missing).** On the dandiset's labels as stored, Anxa1 onset AUC rises with
motif speed (Spearman rho 0.50 to 0.63, p <= 0.0011; fastest quarter of motifs +0.110 against slowest
-0.043 under b(ii)), dLight-dCP rises too, at p < 0.05 in 2 of 4 readings (rho 0.21 to 0.42), and
dLight-cCP falls with speed (rho -0.52 to -0.84, p <= 0.0007; slowest quarter +0.236 against fastest
-0.037). The within-category part cannot be run, as the dandiset holds no category for its motifs,
and the parts it is judged by (fig4f, fig4h) are blocked.


## fig5c-tetx-locomotion

> Anxa1-TeTx mice show moderate reductions in linear and angular speed in the open field, with a
> slight increase in immobility time.

Criterion: per mouse, mean linear speed, mean angular speed and immobility time over the session;
two-sided unpaired t-test against Ctrl-Tdtomato. Reproduced when both speeds are lower at p < 0.05
and immobility is higher; partially reproduced when one speed is lower at p < 0.05. The dandiset's
`linear_velocity` and `angular_velocity` are used. It holds no immobility value and the threshold
is open, so by the claims file's header immobility is not judged.

In [ ]:
O5 = [s for s in oft if s["group"] in anxa1.FIG5_GROUPS]
loco = pd.DataFrame([{"group": s["group"], "subject": s["subject"], "sex": s["sex"], "linear": np.nanmean(s["lin"]),
                      "angular": np.nanmean(s["ang"])} for s in O5])
def ttest_row(ctrl, exp, var):
    a, b = loco[loco.group == ctrl][var], loco[loco.group == exp][var]
    t = stats.ttest_ind(a, b)
    return {"measure": var, "control": ctrl, "group": exp, "n": f"{len(a)}/{len(b)}", "control mean": a.mean(), "group mean": b.mean(),
            "change %": (b.mean() / a.mean() - 1) * 100, "t": t.statistic, "p": t.pvalue,
            "hedges g": (b.mean() - a.mean()) / np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))}
print(pd.DataFrame([ttest_row("Ctrl-Tdtomato", "Anxa1-TeTx", v) for v in ("linear", "angular")]).round(4).to_string(index=False, formatters={"p": "{:.2e}".format}))
print("\nBy sex (linear, angular means):")
print(loco[loco.group.isin(["Ctrl-Tdtomato", "Anxa1-TeTx"])].groupby(["group", "sex"])[["linear", "angular"]].agg(["mean", "size"]).round(3).to_string())
print("\nMann-Whitney variant:", {v: round(stats.mannwhitneyu(loco[loco.group == "Ctrl-Tdtomato"][v], loco[loco.group == "Anxa1-TeTx"][v]).pvalue, 5) for v in ("linear", "angular")})

**Result: reproduced.** Anxa1-TeTx mice are slower than Ctrl-Tdtomato in linear speed (2.81 against
3.93 cm/s, -28.5%, p = 0.0036, Hedges' g -1.13) and in angular speed (0.399 against 0.618 rad/s,
-35.5%, p < 0.0001, g -2.32), in both sexes, and by Mann-Whitney (p = 0.018 and 4e-05). The
increase in immobility is not tested: the dandiset holds no immobility value and the threshold is
open.

## fig5e-early-models-locomotion

> The mild locomotor phenotype and acceleration distribution of Anxa1-silenced mice resemble the
> early-stage PD models (mild 6-OHDA, MP-early), consistent with a bradykinetic and not an
> akinetic/rigid profile.

Criterion: for each experimental group, the change of mean linear speed, mean angular speed and
immobility against its own control (percent of the control mean), and the acceleration density.
Reproduced when the changes of Anxa1-TeTx lie within the range spanned by 6OHDA-low and MP-early, or
closer to theirs than to MP-middle's, for at least two of the three measures. Immobility is not in
the dandiset (fig5c), so two measures are available.

In [ ]:
PAIRS = [("Ctrl-Tdtomato", "Anxa1-TeTx"), ("Ctrl-Asc Acid", "6OHDA-low"), ("Ctrl-early", "MP-early"), ("Ctrl-middle", "MP-middle")]
ch = pd.DataFrame([ttest_row(c, e, v) for c, e in PAIRS for v in ("linear", "angular")])
print(ch.round(4).to_string(index=False, formatters={"p": "{:.2e}".format}))
rows = []
for v in ("linear", "angular"):
    c = ch[ch.measure == v].set_index("group")["change %"]
    lo, hi = sorted([c["6OHDA-low"], c["MP-early"]])
    tet = c["Anxa1-TeTx"]
    d_early, d_mid = min(abs(tet - c["6OHDA-low"]), abs(tet - c["MP-early"])), abs(tet - c["MP-middle"])
    rows.append({"measure": v, "TeTx change %": tet, "early range": f"{lo:.1f} to {hi:.1f}", "inside": lo <= tet <= hi,
                 "distance to nearer early model": d_early, "distance to MP-middle": d_mid, "closer to early": d_early < d_mid})
print(); print(pd.DataFrame(rows).round(2).to_string(index=False))
acc_q = []
for g in anxa1.FIG5_GROUPS:
    a = np.concatenate([s["acc"] for s in O5 if s["group"] == g]); a = a[np.isfinite(a)]
    acc_q.append({"group": g, "p5": np.percentile(a, 5), "p25": np.percentile(a, 25), "median": np.median(a), "p75": np.percentile(a, 75),
                  "p95": np.percentile(a, 95), "fraction |a| > 20 cm/s2": np.mean(np.abs(a) > 20)})
print("\nLinear acceleration distribution (cm/s2), all frames of each group:"); print(pd.DataFrame(acc_q).round(3).to_string(index=False))
fig, ax = plt.subplots(figsize=(6, 3.2))
bins = np.linspace(-60, 60, 121)
for g, col in zip(anxa1.FIG5_GROUPS, plt.cm.tab10.colors):
    a = np.concatenate([s["acc"] for s in O5 if s["group"] == g]); a = a[np.isfinite(a)]
    h, _ = np.histogram(a, bins, density=True); ax.plot(bins[:-1] + 0.5, h, color=col, label=g, lw=1)
ax.set_yscale("log"); ax.set_xlabel("linear acceleration (cm/s2)"); ax.set_ylabel("density"); ax.legend(fontsize=6)
plt.tight_layout(); plt.show()
allacc = np.concatenate([s["acc"] for s in O5]); allacc = allacc[np.isfinite(allacc)]
print(f"frames outside the plotted -60 to 60 cm/s2: {np.mean(np.abs(allacc) > 60):.4f}")

**Result: reproduced.** For both available measures the TeTx change is closer to the early models
than to MP-middle, though outside their range: linear speed -28.5% against -19.8% (6OHDA-low) and
-27.5% (MP-early), 1.0 point from MP-early and 38.2 from MP-middle (-66.7%); angular speed -35.5%
against -19.4% and -14.1%, 16.1 points from the nearer early model and 25.5 from MP-middle (-61.0%).
The acceleration distribution of TeTx mice (5th to 95th percentile -11.3 to 11.3 cm/s2) is narrower
than in the early models (-14.2 to 14.1 and -14.3 to 14.2) and wider than in MP-middle (-7.4 to
7.3). Immobility, the third measure, is not in the dandiset.

## fig5g-pc1-stage

> In a PCA of motif usage, PC1 separates the groups by PD stage, with Anxa1-TeTx mice between the
> early and middle PD models, and PC2 separates middle-stage MitoPark mice from the others.

Criterion: PCA of per-mouse motif usage (fraction of session frames) over all mice of the eight
groups, (a) standardized or (b) centred only; the sign of each PC set so that MP-middle has the
largest group mean. Reproduced when, under a reading, the Anxa1-TeTx mean on PC1 lies between the
early models (6OHDA-low, MP-early) and MP-middle, the controls lie on the low side, and the
MP-middle mean on PC2 lies farther from the mean of all other groups than any other group's mean.
"Between" is read as above both early-model means and below MP-middle; "controls on the low side"
as every control mean below every experimental group's mean. Labels never used by these mice are
left out (they have no variance).

Blocked (data missing), as it rests on the paper's motif labelling (fig4c). The test below runs on
the dandiset's own labels and is shown, not judged.


In [ ]:
U_all = np.stack([np.bincount(s["motif"], minlength=NM) / len(s["motif"]) for s in O5])
grp5 = np.array([s["group"] for s in O5])
mused = np.flatnonzero(U_all.sum(0) > 0)
U5 = U_all[:, mused]
spd5 = motif_speed(O5)[mused]
PCA = {}
rows = []
for sc in ("a standardized", "b centred"):
    X = U5 - U5.mean(0)
    if sc.startswith("a"):
        X = X / U5.std(0, ddof=1)
    u, sv, vt = np.linalg.svd(X, full_matrices=False)
    scores, load = u * sv, vt.copy()
    for pc in range(4):
        if pd.Series(scores[:, pc]).groupby(grp5).mean()["MP-middle"] < 0:
            scores[:, pc] *= -1; load[pc] *= -1
    PCA[sc] = (scores, load, sv ** 2 / np.sum(sv ** 2))
    gm = pd.DataFrame({"PC1": pd.Series(scores[:, 0]).groupby(grp5).mean(), "PC2": pd.Series(scores[:, 1]).groupby(grp5).mean()}).loc[anxa1.FIG5_GROUPS]
    print(f"{sc}: {len(mused)} motifs, {len(O5)} mice, variance explained PC1-4 {np.round(PCA[sc][2][:4], 3).tolist()}")
    print(gm.round(3).T.to_string())
    ctrls, exps = ["Ctrl-Tdtomato", "Ctrl-Asc Acid", "Ctrl-early", "Ctrl-middle"], ["Anxa1-TeTx", "6OHDA-low", "MP-early", "MP-middle"]
    between = gm.PC1["Anxa1-TeTx"] > max(gm.PC1["6OHDA-low"], gm.PC1["MP-early"]) and gm.PC1["Anxa1-TeTx"] < gm.PC1["MP-middle"]
    low = gm.PC1[ctrls].max() < gm.PC1[exps].min()
    dist = {g: abs(gm.PC2[g] - gm.PC2.drop(g).mean()) for g in anxa1.FIG5_GROUPS}
    pc2 = max(dist, key=dist.get) == "MP-middle"
    t_pc1 = stats.ttest_ind(scores[grp5 == "Anxa1-TeTx", 0], scores[grp5 == "Ctrl-Tdtomato", 0])
    rows.append({"scaling": sc, "TeTx between early and middle": between, "controls low": low, "MP-middle farthest on PC2": pc2,
                 "PC2 distances": {k: round(v, 2) for k, v in dist.items()}, "TeTx vs Ctrl-Tdtomato PC1 t-test p": t_pc1.pvalue})
print(pd.DataFrame(rows).drop(columns="PC2 distances").to_string(index=False))
for r in rows: print(r["scaling"], "PC2 distance of each group mean from the mean of the others:", r["PC2 distances"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, sc in zip(axes, PCA):
    sco = PCA[sc][0]
    for g, col in zip(anxa1.FIG5_GROUPS, plt.cm.tab10.colors):
        ax.scatter(sco[grp5 == g, 0], sco[grp5 == g, 1], s=10, color=col, label=g)
    ax.set_title(sc); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
axes[0].legend(fontsize=6); plt.tight_layout(); plt.show()
for sc in PCA: print(sc, "plotted PC1", PCA[sc][0][:, 0].min().round(2), "to", PCA[sc][0][:, 0].max().round(2), "| PC2", PCA[sc][0][:, 1].min().round(2), "to", PCA[sc][0][:, 1].max().round(2))

**Result: blocked (data missing).** On the dandiset's labels, with standardized usage, the
Anxa1-TeTx mean on PC1 (3.39) lies above the early models (6OHDA-low -0.32, MP-early -0.48) and below
MP-middle (5.25), every control mean (-2.94 to -0.81) is below every experimental mean, and on PC2
MP-middle (2.15) lies 2.41 from the mean of the other groups, the largest distance (next 1.45). With
centred usage the first two parts hold and the PC2 part does not (MP-middle 0.01). The PCA is over 39
dandiset labels and 108 mice, not the paper's 37 motifs. Usage does not depend on where the labels
sit on the frame grid.


## fig5i-pc1-motif-speed

> Motif PC1 values correlate negatively with mean motif speed (R = -0.762), while PC2 shows no
> significant correlation (R = -0.1494, p = 0.3772), nor do PC3 and PC4.

Criterion: per motif, the PC1 to PC4 loading against mean motif linear speed over all mice (here
the mice of the PCA); Pearson R. The magnitude is judged with PC1 oriented as in fig5g. Reproduced
when, under a reading of fig5g, |R| for PC1 is within 0.762 +/- 0.0762 with p < 0.0001, and PC2,
PC3 and PC4 each have p >= 0.05.

Blocked (data missing), as it rests on the paper's motif labelling (fig4c). The test below runs on
the dandiset's own labels and is shown, not judged.


In [ ]:
rows = []
for sc, (scores, load, ev_) in PCA.items():
    for pc in range(4):
        r = stats.pearsonr(load[pc], spd5)
        rows.append({"scaling": sc, "PC": pc + 1, "R": r[0], "p": r[1]})
I5 = pd.DataFrame(rows); print(I5.round(5).to_string(index=False))
for sc in PCA:
    g = I5[I5.scaling == sc].set_index("PC")
    print(sc, "| PC1 |R| within tolerance and p < 0.0001:", abs(abs(g.R[1]) - 0.762) <= 0.0762 and g.p[1] < 1e-4,
          "| PC2-4 all p >= 0.05:", bool((g.p[[2, 3, 4]] >= 0.05).all()))
spd5_best = motif_speed(O5, BEST_SHIFT)[mused]
print(f"\nWith the labels moved {BEST_SHIFT:+d} frames for motif speed (label timing check):",
      {sc: [round(stats.pearsonr(PCA[sc][1][pc], spd5_best)[0], 3) for pc in range(4)] for sc in PCA}, "(R of PC1 to PC4)")


**Result: blocked (data missing).** On the dandiset's labels as stored, PC1 loadings correlate
negatively with mean motif speed, R = -0.640 (p = 1.2e-05) with standardized usage and -0.477 (p =
0.0021) with centred usage, both weaker than -0.762 +/- 0.076. The other PCs are not all unrelated to
speed: with standardized usage PC2 (R = -0.329, p = 0.041) and PC4 (R = 0.347, p = 0.031) correlate,
and with centred usage PC2 does (R = -0.425, p = 0.0069); PC3 does not in either (p = 0.79 and 0.13).
With motif speed taken with the labels moved +18 frames, where they explain speed best, PC1 has R =
-0.774 with standardized usage (PC2 to PC4 R = -0.19, 0.19 and 0.29) and -0.433 centred.


### Motif numbers (not deciding)

fig5j-tetx-five-motifs, fig5j-shared-and-stage-motifs and fig5m-walk-motif-network rest on the
paper's motif numbers, which the dandiset's labels are not (fig4c), so they are blocked, as are
all the motif claims. The same
tests run on the dandiset's labels show what the labels give; they decide nothing.

In [ ]:
lab_rows = []
for c, e in PAIRS:
    a, b = U5[grp5 == c], U5[grp5 == e]
    p = np.array([stats.ttest_ind(a[:, j], b[:, j]).pvalue for j in range(len(mused))])
    rej, padj, _, _ = multipletests(p, alpha=0.05, method="holm-sidak")
    lab_rows.append({"group": e, "labels down (adjusted p < 0.05)": [int(mused[j]) for j in np.flatnonzero(rej & (b.mean(0) < a.mean(0)))],
                     "labels up": [int(mused[j]) for j in np.flatnonzero(rej & (b.mean(0) > a.mean(0)))]})
print(pd.DataFrame(lab_rows).to_string(index=False))
print("Mean speed of the labels down in Anxa1-TeTx (cm/s):", {int(k): round(float(spd5[list(mused).index(k)]), 2) for k in lab_rows[0]["labels down (adjusted p < 0.05)"]})

## fig6c-maze-locomotion

> In the arrow maze, Anxa1-silenced mice show only a small decrease in linear speed on the fourth
> day and no change in immobility time.

Criterion: per mouse and day (days 1 and 4), mean linear speed over the 10 min session; RM two-way
ANOVA group by day with Sidak's post hoc. Reproduced when the group effect has p < 0.05 with
Anxa1-TeTx slower, the Sidak test is p < 0.05 on day 4 only, and immobility shows no group effect or
Sidak difference. The dandiset holds pose only, so speed is computed under the header's readings,
(a) the tail base and (b) the mean of the six points (snout, four paws, tail base), in pixels per s
(one camera, so the scale does not change a test). Smoothing is open; raw positions and positions
with DeepLabCut confidence below 0.9 interpolated are both run. Immobility has no value in the
dandiset or the paper, so it is not judged. Sidak's test compares the groups by an unpaired t-test
on each day, adjusted over the two days (a standard implementation; Prism pools the error term).

In [ ]:
mrows = []
for s in [s for s in maze if s["cohort"] == "tmaze_anxa1_tet"]:
    for pts in ("tail", "mean6"):
        for mc in (None, 0.9):
            v = anxa1.maze_speed(s, pts, mc)
            mrows.append({"subject": s["subject"], "group": s["group"], "day": s["day"], "points": pts,
                          "cleaning": "raw" if mc is None else "confidence 0.9", "speed": float(np.nanmean(v))})
MZ = pd.DataFrame(mrows)
print("Mean speed (px/s) by group and day:")
MZM = MZ.groupby(["points", "cleaning", "group", "day"]).speed.mean().unstack()
print(MZM.round(1).to_string())
print("Mean of six points over tail base, group means:")
print((MZM.loc["mean6"] / MZM.loc["tail"]).round(2).to_string())
rows = []
for (pts, cl), g in MZ[MZ.day.isin([1, 4])].groupby(["points", "cleaning"]):
    a = anxa1.mixed_anova(g, "speed", "day", "group", "subject").set_index("Source")
    ps = {}
    for d in (1, 4):
        x, y = g[(g.day == d) & (g.group == "Ctrl-Tdtomato")].speed, g[(g.day == d) & (g.group == "Anxa1-TeTx")].speed
        ps[d] = (anxa1.sidak(stats.ttest_ind(x, y).pvalue, 2), (y.mean() / x.mean() - 1) * 100)
    slower = g[g.group == "Anxa1-TeTx"].speed.mean() < g[g.group == "Ctrl-Tdtomato"].speed.mean()
    rows.append({"points": pts, "cleaning": cl, "group F": a.loc["group", "F"], "group df": f"{a.loc['group', 'DF1']}, {a.loc['group', 'DF2']}",
                 "group p": a.loc["group", "p_unc"], "interaction p": a.loc["Interaction", "p_unc"], "TeTx slower": slower,
                 "Sidak day 1": ps[1][0], "change day 1 %": ps[1][1], "Sidak day 4": ps[4][0], "change day 4 %": ps[4][1],
                 "holds": a.loc["group", "p_unc"] < 0.05 and slower and ps[4][0] < 0.05 and ps[1][0] >= 0.05})
print(); print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nMice:", MZ.drop_duplicates("subject").groupby("group").size().to_dict())

**Result: not reproduced.** Under none of the four speed readings is the day 4 difference
significant (Sidak p = 0.11 to 0.41). A group effect with Anxa1-TeTx slower appears only with the
tail base and low-confidence positions interpolated (F = 4.98, p = 0.035; -21.8% on day 1 and -17.7%
on day 4); with raw tail base positions there is none (p = 0.75), and with the mean of six points
TeTx mice are faster (raw, p = 0.021) or not different (p = 0.44). Immobility is not tested. The
dandiset has 17 control mice against the legend's 18. The tail base occupies the same pixel extent on
every day (medians of the 1st and 99th percentiles 98 to 111 and 567 to 620 px in x, 185 to 232
and 699 to 706 px in y), so the scale does
not change between days. The speeds are unsmoothed apart from the interpolation of low-confidence
positions, and the mean of six points is noisy (raw speeds 1.5 to 5.9 times those of the tail base in the group means, and
a jump on day 4 with low-confidence positions interpolated); the claims file leaves smoothing open, so no other smoothing is applied, and the
outcome rests on these noisy measures.

## fig6j-reaching-accuracy

> Control mice improve reaching accuracy over four sessions; Anxa1-TeTx mice have normal accuracy in
> the first session but fail to improve, and Anxa1-Arch mice show reduced accuracy during the last
> three sessions with optogenetic inhibition.

Criterion: per mouse and session, the fraction of grasp attempts with the preferred forelimb whose
distance from the target is within a threshold, < 0.2 cm (Results) or < 0.1 cm (legend of 6J); RM
two-way ANOVA group by day on days 1 and 4 with Sidak's post hoc. Reproduced when, under a reading,
the interaction has p < 0.05, controls improve from day 1 to day 4 at Sidak p < 0.05, controls
exceed both Anxa1-TeTx and Anxa1-Arch on day 4 at Sidak p < 0.05, and Anxa1-TeTx does not differ
from controls on day 1; each single exclusion of a mouse is shown as a variant.

After the answered questions the criterion fixes the rest. Grasp attempts are (a) every labelled
reach with a distance, or (b) all but `empty` and `unknown`; the preferred forelimb is the paw used
most (i) per mouse over all its sessions, or (ii) per session. Day 1 and day 4 are the first and
fourth training sessions of each mouse; the files label them `test1` and `test4`, and their dates
are checked below. For Arch, a day 1 or day 4 session whose laser label departs from the Methods
(off on day 1, on on days 2 to 4) is reported. Controls of both cohorts are one pooled group. The
main run uses the mice recorded before the preprint (22 December 2024); all mice with both days are
a variant. Sessions of one mouse on one day are pooled. Sidak's tests: controls day 1 against day
4 (paired t-test, Sidak over the three groups), and controls against each silenced group on each
day (unpaired t-tests, Sidak over the four comparisons).

In [ ]:
RS = [s for s in reach if s["experiment"] in ("reaching_test_anxa1_tet", "reaching_test_anxa1_arch")]
SESS = pd.DataFrame([{"subject": s["subject"], "group": s["group"], "cohort": "TeTx" if "tet" in s["experiment"] else "Arch",
                      "phase": s["phase"], "start": s["start"][:16], "events": len(s["events"]), "light": s["light"]} for s in RS])
SESS["date"] = pd.to_datetime(SESS.start.str[:10])
first = SESS.groupby("subject").date.transform("min")
SESS["day of training"] = (SESS.date - first).dt.days + 1
print("Sessions by test label: files, mice, and day of training counted from each mouse's first session")
print(SESS.groupby(["cohort", "phase"]).agg(files=("subject", "size"), mice=("subject", "nunique"), with_events=("events", lambda e: int((e > 0).sum())),
                                           day_min=("day of training", "min"), day_max=("day of training", "max")).to_string())
print("Mice with two files on one day (pooled):", SESS[SESS.duplicated(["subject", "phase"], keep=False)].subject.unique().tolist())
print("Mice whose test4 is not 3 days after test1:",
      SESS[SESS.phase == "test4"].query("`day of training` != 4")[["subject", "cohort", "day of training"]].values.tolist())
arch = SESS[(SESS.cohort == "Arch") & SESS.phase.isin(["test1", "test4"])]
print("\nArch cohort light labels on days 1 and 4 (Methods: off on day 1, on on days 2 to 4):")
print(arch.groupby(["phase", "light"]).agg(files=("subject", "size"), mice=("subject", "nunique")).to_string())

RE = []
for s in RS:
    e = s["events"].copy()
    e["subject"], e["group"], e["phase"], e["date"] = s["subject"], s["group"], s["phase"], s["start"][:10]
    e["cohort"] = "TeTx" if "tet" in s["experiment"] else "Arch"
    RE.append(e)
RE = pd.concat(RE, ignore_index=True)
RE = RE[RE.paw.isin(["left", "right"])]
SETS = {"a every labelled reach": ["correct", "drop", "semidrop", "miss", "empty", "unknown"], "b no empty or unknown": ["correct", "drop", "semidrop", "miss"]}
PREPRINT = "2024-12-22"

def accuracy(sets, thr, pref="mouse", before=PREPRINT):
    X = RE[RE.label.isin(sets) & RE.phase.isin(["test1", "test4"]) & RE.distance_cm.notna()].copy()
    if before:
        X = X[X.groupby("subject").date.transform("min") < before]
    if pref == "mouse":
        p = RE.groupby("subject").paw.agg(lambda x: x.value_counts().idxmax())
        X = X[X.paw == X.subject.map(p)]
    else:
        p = X.groupby(["subject", "phase"]).paw.agg(lambda x: x.value_counts().idxmax())
        X = X[X.paw.values == p.loc[list(zip(X.subject, X.phase))].values]
    a = X.groupby(["subject", "group", "cohort", "phase"]).agg(acc=("distance_cm", lambda d: (d < thr).mean()), n=("distance_cm", "size")).reset_index()
    both = a.groupby("subject").phase.nunique()
    a = a[a.subject.isin(both[both == 2].index)]
    a["day"] = a.phase.map({"test1": 1, "test4": 4})
    return a

In [ ]:
def reach_tests(a):
    an = anxa1.mixed_anova(a, "acc", "day", "group", "subject").set_index("Source")
    g = lambda grp, d: a[(a.group == grp) & (a.day == d)].set_index("subject").acc
    c1, c4 = g("Ctrl", 1), g("Ctrl", 4)
    within = {grp: stats.ttest_rel(g(grp, 1), g(grp, 4).loc[g(grp, 1).index]).pvalue for grp in ["Ctrl", "Anxa1-TeTx", "Anxa1-Arch"]}
    between = {(grp, d): stats.ttest_ind(g("Ctrl", d), g(grp, d)).pvalue for grp in ["Anxa1-TeTx", "Anxa1-Arch"] for d in (1, 4)}
    out = {"mice": a.drop_duplicates("subject").groupby("group").size().to_dict(),
           "interaction F": an.loc["Interaction", "F"], "df": f"{an.loc['Interaction', 'DF1']}, {an.loc['Interaction', 'DF2']}",
           "interaction p": an.loc["Interaction", "p_unc"],
           "ctrl d1": c1.mean(), "ctrl d4": c4.mean(), "tet d1": g("Anxa1-TeTx", 1).mean(), "tet d4": g("Anxa1-TeTx", 4).mean(),
           "arch d1": g("Anxa1-Arch", 1).mean(), "arch d4": g("Anxa1-Arch", 4).mean(),
           "Sidak ctrl d1 vs d4": anxa1.sidak(within["Ctrl"], 3), "Sidak ctrl vs tet d4": anxa1.sidak(between[("Anxa1-TeTx", 4)], 4),
           "Sidak ctrl vs arch d4": anxa1.sidak(between[("Anxa1-Arch", 4)], 4), "Sidak ctrl vs tet d1": anxa1.sidak(between[("Anxa1-TeTx", 1)], 4),
           "Sidak ctrl vs arch d1": anxa1.sidak(between[("Anxa1-Arch", 1)], 4)}
    out["holds"] = (out["interaction p"] < 0.05 and c4.mean() > c1.mean() and out["Sidak ctrl d1 vs d4"] < 0.05
                    and out["ctrl d4"] > out["tet d4"] and out["Sidak ctrl vs tet d4"] < 0.05
                    and out["ctrl d4"] > out["arch d4"] and out["Sidak ctrl vs arch d4"] < 0.05 and out["Sidak ctrl vs tet d1"] >= 0.05)
    return out

rows = []
for before in (PREPRINT, None):
    for sn, sets in SETS.items():
        for pref in ("mouse", "session"):
            for thr in (0.2, 0.1):
                rows.append({"mouse set": "before preprint" if before else "all", "events": sn[0], "preferred limb": pref, "threshold cm": thr,
                             **reach_tests(accuracy(sets, thr, pref, before))})
RJ = pd.DataFrame(rows)
cols = ["events", "preferred limb", "threshold cm", "mice", "interaction F", "df", "interaction p", "ctrl d1", "ctrl d4", "tet d1", "tet d4",
        "arch d1", "arch d4", "Sidak ctrl d1 vs d4", "Sidak ctrl vs tet d4", "Sidak ctrl vs arch d4", "Sidak ctrl vs tet d1", "holds"]
print("Main run, mice recorded before the preprint (2024-12-22):")
print(RJ[RJ["mouse set"] == "before preprint"][cols].round(4).to_string(index=False))
print("\nVariant, all mice with both days:")
print(RJ[RJ["mouse set"] == "all"][cols].round(4).to_string(index=False))

In [ ]:
print("\nMice with annotated reaches on both days, before the preferred limb is applied:")
for before in (PREPRINT, None):
    X = RE[RE.phase.isin(["test1", "test4"]) & RE.distance_cm.notna()]
    if before:
        X = X[X.groupby("subject").date.transform("min") < before]
    nd = X.groupby("subject").phase.nunique()
    X = X[X.subject.isin(nd[nd == 2].index)]
    kept = set(accuracy(SETS["a every labelled reach"], 0.2, "mouse", before).subject)
    print(f"  {'before preprint' if before else 'all'}:", X.drop_duplicates("subject").groupby("group").size().to_dict(),
          "| dropped by the per-mouse preferred limb (no reach with it on a day):", sorted(set(X.subject) - kept))
for before in (PREPRINT, None):
    a = accuracy(SETS["a every labelled reach"], 0.2, "mouse", before)
    print(f"  {'before preprint' if before else 'all'}, events a: reaches per mouse and day median", int(a.n.median()), "min", int(a.n.min()))
for pref in ("mouse", "session"):
    base = accuracy(SETS["a every labelled reach"], 0.2, pref)
    loo = []
    for sub in base.subject.unique():
        r = reach_tests(base[base.subject != sub])
        loo.append({"left out": sub, "group": base[base.subject == sub].group.iloc[0], "interaction p": r["interaction p"],
                    "Sidak ctrl vs arch d4": r["Sidak ctrl vs arch d4"], "holds": r["holds"]})
    loo = pd.DataFrame(loo)
    print(f"\nSingle exclusions (main set, events a, 0.2 cm, preferred limb per {pref}): criterion holds in {loo.holds.sum()} of {len(loo)}; "
          f"interaction p {loo['interaction p'].min():.4f} to {loo['interaction p'].max():.4f}; Arch day 4 Sidak p {loo['Sidak ctrl vs arch d4'].min():.4f} to {loo['Sidak ctrl vs arch d4'].max():.4f}")
    print(loo[loo.holds].round(4).to_string(index=False) if loo.holds.any() else "no single exclusion makes it hold")
a = accuracy(SETS["a every labelled reach"], 0.2, "session")
print("\nPer mouse accuracy (main set, events a, preferred limb per session, < 0.2 cm):")
print(a.pivot_table(index=["group", "cohort", "subject"], columns="day", values="acc").round(3).to_string())

**Result: partially reproduced.** In the main run, the 17 mice recorded before the preprint (8
controls, 4 TeTx and 5 Arch; 8 + 4 + 5 fits the legend's F(2,14)), the control and TeTx parts hold
and the Arch part does not. With every labelled reach, the preferred limb per mouse and 0.2 cm, the
interaction has F(2, 13) = 4.42, p = 0.034, controls improve from 0.338 to 0.650 (Sidak p = 0.010),
controls exceed TeTx on day 4 (0.239, p = 0.0014) and do not differ from them on day 1 (p = 0.994),
but Arch on day 4 (0.428) does not differ from controls (p = 0.22); one Arch mouse (842433) has no
reach with its preferred limb on one day under this reading and drops out. Without empty and
unknown reaches the same parts hold (interaction p = 0.046, control improvement p = 0.020, TeTx
below controls p = 0.016) and Arch fails (p = 0.34). With the preferred limb per session all 17 mice
enter and the df are the legend's (2, 14), but the interaction misses (p = 0.060, and 0.085 without
empty and unknown reaches). The Arch day 4 comparison fails in all eight readings of the main run
(Sidak p = 0.044 to 0.34; the one below 0.05, at 0.1 cm, comes with an interaction p of 0.12), and
with 0.1 cm the interaction misses in every reading (p = 0.069 to 0.27). Of the single exclusions,
none of 16 makes the criterion hold with the limb per mouse and one of 17 with the limb per session
(leaving out Arch mouse 828027: interaction p = 0.016, Arch day 4 p = 0.035). The variant with all
mice holds at 0.2 cm under all four event and limb readings (interaction p = 0.0030 to 0.028). This
departs from the paper's protocol, as the claims file asks to report: in the Arch cohort all 17 mice,
controls included, have their day 1 session labelled with the laser switched on after the first
reach, where the Methods have it off on day 1; these sessions are used as day 1. Day 4 is labelled
laser on, as in the Methods. The outcome is partial and not a failure because each part points the
claimed way: on day 4 Arch accuracy is below the controls' in all eight readings of the main run
(0.099 to 0.428 against 0.264 to 0.650), but not at Sidak p < 0.05 with an interaction at p < 0.05;
the verifier judged it not reproduced, and that disagreement is recorded.
The sessions labelled `test4` fall 2 to 4 days after `test1`.

In [ ]:
print(f"Run time of the notebook to here: {(time.time() - started) / 60:.1f} min")


## Summary

| Claim | Outcome |
|---|---|
| figs2-evoked-da-dcp-ccp | reproduced (all powers pooled; each power alone holds at 3 or 4 of 5) |
| fig3d-anxa1-ramp-to-touch | not reproduced |
| fig3e-dlight-dcp-ccp-reward | not reproduced (cCP part holds, dCP part fails in every reading) |
| fig3fg-auc-change-from-baseline | partially reproduced (holds with the baseline z-score, not the session z-score) |
| fig4b-speed-correlation-dcp | not reproduced (dLight-dCP holds; Anxa1 positive in 9 of 10 but p = 0.064 to 0.21 in every reading) |
| fig4b-speed-correlation-ccp-negative | reproduced |
| fig5c-tetx-locomotion | reproduced (immobility not tested) |
| fig5e-early-models-locomotion | reproduced (two of three measures) |
| fig6c-maze-locomotion | not reproduced |
| fig6j-reaching-accuracy | partially reproduced (controls and TeTx hold, Arch lower but not significantly) |
| fig4c-vame-motifs, fig4e-anxa1-high-speed-motifs, fig4f-auc-correlations, fig4h-auc-motif-speed, fig5g-pc1-stage, fig5i-pc1-motif-speed | blocked: data missing (the paper's motif labelling is not in the dandiset) |
| fig5h-walk-stationary-loadings, fig5j-tetx-five-motifs, fig5j-shared-and-stage-motifs, fig5m-walk-motif-network | blocked: data missing (as above) |
| fig6d-maze-preference-learning, fig6d-maze-efficiency | blocked: data missing (no end-zone entries or rewards) |
| fig1b-anxa1-downregulated-rnaseq | blocked: other (GSE282753 lacks the cMfn2KO data; the MitoPark half holds) |
| fig1c, fig1e, fig1h, fig1j, fig1m, figs1-6ohda, fig1p, figs1-vta, fig2c, fig2f, fig2i, fig2n | blocked: data missing |
| fig3i, fig3kl, fig3o | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The photometry of the water drop and evoked release
sessions is z-scored here under the claims file's two readings, from the dandiset's dF/F; the
paper's DoricStudio z-scoring is not known. The open field analyses use the dandiset's own 30 Hz
z-scored trace and kinematics. The motif claims are blocked, as the dandiset's VAME labels are not
the paper's; their analyses are shown on the dandiset's labels and decide nothing. Every recording
of a group is used where the dandiset holds more than the legends count. The arrow maze speed is
computed from pose, unsmoothed, since the dandiset has no kinematics for it. The water drop touch and
the grasp attempts are read from the annotated event labels in two ways each, the readings the
answered questions named. The RNA-seq of the MitoPark half is tested on the GEO counts with
pydeseq2 and with edgepy, a Python port of edgeR, with TMM written out in the helper and the GLM
dispersion estimators (the port has no estimateDisp), not with the R packages; libraries are summed
per mouse. Prism's Sidak tests after the ANOVAs are approximated by t-tests with Sidak adjustment,
without Prism's pooled error term.

What was found about the dataset:
- It holds only behaviour and photometry: no histology, RNA-seq, qPCR or nosepoke counts, so 16
  claims of Figures 1 to 3 cannot be tested; no file holds a nosepoke, reward, end-zone or trial
  table, and the arrow maze has pose only, without the Bonsai end-zone entries, rewards or arena
  geometry.
- 86 open field files are named as `ses-tmaze` sessions (session_id `tmaze-...`), and groups,
  cohorts, days and test phases are given only in free text in `session_description`.
- Subject genotypes conflict across files (dLight mice are WT in their open field files and
  Anxa1-flp or Dat_cre elsewhere; 687662 is anxa1-flp and Dat_cre), and mouse 779575 is a control
  in the open field TeTx experiment and a TeTx mouse in the reaching cohort.
- The VAME labels (42) are not the paper's 37 numbered motifs and carry no categories. They are
  stored with `starting_time` 0, but they explain speed best when placed about 18 frames (0.6 s)
  later, half the VAME window, so motif onsets as stored come early; this is not documented.
- The z-scored open field photometry is smoothed and lags the dF/F by about 0.1 s, neither
  documented; seven recordings hold under 13 minutes of photometry.
- The dandiset holds more recordings and mice than the legends in the photometry and reaching
  experiments, and experiments not in the preprint (6-OHDA in tail striatum, anti-Anxa1, a 2025 Arch
  cohort), without saying which are the paper's.
- Every Arch-cohort reaching file labels day 1 with the laser switched on after the first reach,
  where the Methods have the laser off on day 1.
- GEO GSE282753 holds the MitoPark RNA-seq counts and RPKM (5 MitoPark, 7 control mice) but not the
  cMfn2KO counts.
- The evoked release sessions are described as nosepoke self-stimulation, but the trains come every
  4 s; the session descriptions call the photometry GCaMP signals SN cell-body recordings while the
  fiber table places them in dCP.
